In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *
from pyspark.sql.window import *

**Q1 While ingesting customer data from an external source, you notice duplicate entries. How would you remove duplicates and retain only the latest entry based on a timestamp column?**

In [0]:
data = [("101", "2023-12-01", 100), ("101", "2023-12-02", 150), 
        ("102", "2023-12-01", 200), ("102", "2023-12-02", 250)]
columns = ["product_id", "date", "sales"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
## Casting Date column from String to Date type

df = df.withColumn("date", to_date('date'))
display(df)


In [0]:
#By using Window Function

df = df.withColumn('flag',dense_rank().over(Window.partitionBy('product_id').orderBy(col('date').desc()))).filter(col('flag')==1).drop('flag')
display(df)

In [0]:
#Without using Window function

df = df.orderBy('product_id','date', ascending=[1,0]).dropDuplicates(subset=['product_id']).display() 

**2. While processing data from multiple files with inconsistent schemas, you need to merge them into a single DataFrame. How would you handle this inconsistency in PySpark?**

In [0]:
df = spark.read.format("parquet")\
    .option("mergeSchema",true)\
        .load("/Files/Data/DataFiles")

**4. You are working with a real-time data pipeline, and you notice missing values in your streaming data Column - Category. How would you handle null or missing values in such a scenario?**

df_stream = spark.readStream.schema("id INT, value STRING").csv("path/to/stream")

In [0]:
df = df.fillNa({'<column_name>': '<value>'})
df = df.fillNa({'Category': 'UnKnown'})

**5. You need to calculate the total number of actions performed by users in a system. How would you calculate the top 5 most active users based on this information?**

In [0]:
data = [("user1", 5), ("user2", 8), ("user3", 2), ("user4", 10), ("user2", 3)]
columns = ["user_id", "actions"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
#Change Data Type of col actions from String to Int

df = df.withColumn("actions",col("actions").cast(IntegerType()))
df.display()

In [0]:
df = df.groupBy("user_id").agg(sum("actions").alias("total_actions")).orderBy('total_actions', ascending=[0]).limit(3)

display(df)


**6. While processing sales transaction data, you need to identify the most recent transaction for each customer. How would you approach this task?**

In [0]:
data = [("cust1", "2023-12-01", 100), ("cust2", "2023-12-02", 150),
        ("cust1", "2023-12-03", 200), ("cust2", "2023-12-04", 250)]
columns = ["customer_id", "transaction_date", "sales"]
df = spark.createDataFrame(data, columns)
df.display()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

df = df.withColumn('transaction_date', to_date(col('transaction_date')))

display(df)

In [0]:
#Change data type of col 'transaction_date' to dateType

df = df.withColumn("transaction_date", col("transaction_date").cast(DateType()))

In [0]:
from pyspark.sql.window import *

In [0]:
# Apply window function to rank transactions by date (most recent = rank 1)
df = df.withColumn('flag', dense_rank().over(Window.partitionBy('customer_id').orderBy(col('transaction_date').desc())))\
    .filter(col('flag')==1)
display(df)

**# 7. You need to identify customers who haven’t made any purchases in the last N days.
How would you filter such customers?**

In [0]:
data = [("cust1", "2026-12-01"), ("cust2", "2025-11-20"), ("cust3", "2025-11-25")]
columns = ["customer_id", "last_purchase_date"]

df = spark.createDataFrame(data, columns)

df.display()

In [0]:
df = df.withColumn('gap',date_diff(current_date(),'last_purchase_date'))#.filter(col('gap')>200)

display(df)

**8. While analyzing customer reviews, you need to identify the most frequently used words in the feedback. How would you implement this?**

In [0]:
data = [("customer1", "The product is great"), ("customer2", "Great product, fast delivery"), ("customer3", "Not bad, could be better")]
columns = ["customer_id", "feedback"]

df = spark.createDataFrame(data, columns)

df.display()

In [0]:
df = df.withColumn('feedback',explode(split(lower('feedback'),' ')))

display(df)

In [0]:
df = df.groupBy("feedback").agg(count("feedback").alias("w_Count")).orderBy(col('w_Count').desc())\
.filter(col('w_Count')>1)

display(df)

**9. You need to calculate the cumulative sum of sales over time for each product. How would you approach this?**

In [0]:
data = [("product1", "2023-12-01", 100), ("product2", "2023-12-02", 200),
        ("product1", "2023-12-03", 150), ("product2", "2023-12-04", 250)]
columns = ["product_id", "date", "sales"]
df = spark.createDataFrame(data, columns)
df.display()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

df = df.withColumn('date',to_date('date'))


In [0]:


df = df.withColumn('cum_sales',sum('sales').over(Window.partitionBy('product_id').orderBy(col('date').asc())))

df = df.select("product_id","date","sales","cum_sales")

display(df)

**10. While preparing a data pipeline, you notice some duplicate rows in a dataset. How would you remove the duplicates without affecting the original order?**

In [0]:
data = [("John", 25), ("Jane", 30), ("John", 25), ("Alice", 22)]
columns = ["name", "age"]
df = spark.createDataFrame(data, columns)
df.display()

In [0]:
df = df.dropDuplicates(subset=['name','age'])
display(df)

In [0]:
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

df = df.withColumn('key',row_number().over(Window.partitionBy("name","age").orderBy(col('age')))).filter(col('key')==1)

display(df)

**11. You are working with user activity data and need to calculate the average session duration per user. How would you implement this?**

In [0]:
data = [("user1", "2023-12-01", 50), ("user1", "2023-12-02", 60), 
        ("user2", "2023-12-01", 45), ("user2", "2023-12-03", 75)]
columns = ["user_id", "session_date", "duration"]
df = spark.createDataFrame(data, columns)

df.display()

In [0]:
df = df.groupBy("user_id").agg(avg("duration").alias("avg_dur"))
display(df)

**12. While analyzing sales data, you need to find the product with the highest sales for each month. How would you accomplish this?**

In [0]:
data = [("product1", "2023-12-01", 100), ("product2", "2023-12-01", 150), 
        ("product1", "2023-12-02", 200), ("product2", "2023-12-02", 250)]
columns = ["product_id", "date", "sales"]
df = spark.createDataFrame(data, columns)
df.display()

In [0]:
df = df.withColumn('date',to_date('date'))

display(df)

In [0]:
df = df.withColumn('date',month('date')).groupBy("product_id","date").agg(sum("sales").alias("total_sales"))

#display(df)

df = df.withColumn('flag',dense_rank().over(Window.partitionBy('date').orderBy(col('total_sales').desc()))).filter(col('flag')==1).drop('flag')

display(df)

**# What is the role of SparkContext in PySpark**

Ans : The SparkContext serves as the entry point for Spark.
It acts as a bridge between the driver program and the cluster manager, establishing the necessary connection to the cluster.
This connection allows Spark to access machine resources such as CPU and memory, enabling the processing of data

# Explain Spark Architecture
Apache Spark is a kind of framework that help to perform Distrbutred computing.

**PySaprk is a Python Wrapper on TOP of Apache Spark**

-When an application is submitted, it first goes to the Cluster Manager.

**-Driver Program:** The Cluster Manager selects a machine within the cluster to act as the Driver Node. Within this node, the Spark Context is created, which serves as the "bridge" or entry point to manage the connection with the cluster.

**-Resource Allocation:** The Driver Node analyzes the code and determines the required resources. It communicates with the Cluster Manager via the Spark Context to request and assign Worker Nodes to perform the processing.

**-Execution:** Once the Worker Nodes are assigned, the Driver Node (acting as the "brain") communicates the necessary transformations and instructions to them, and the workers execute these tasks in parallel.

**Note on Deployment Modes:** If the program is running in cluster mode, the driver node resides inside the cluster. However, in client mode, the driver node resides outside of the cluster, such as on a local machine or a separate virtual machine 

![image_1785215585917.png](./image_1785215585917.png "image_1785215585917.png")![]

**13. You are working with a large Delta table that is frequently updated by multiple users. The data is stored in partitions, and sometimes updates can cause inconsistent reads due to concurrent transactions. How would you ensure ACID compliance and avoid data corruption in PySpark?**

-First We need to create a Delta log table where all the transactions will be recorded.
-Secondly we can upsert or merge the incoming data with the target on the basis of any id column.




In [0]:
#Read New Data

df = spark.read.format('parquet').load('path')

# import DeltaLiveTable

from delta.tables import *
#creating an Object on top of the residing Data
dlt_table = DeltaTable.forPath('path') #Targert


dlt_table.alias('trg').merge(df.alias('src'), 'trg.id = src.id').whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()





**# 14. You need to process a large dataset stored in PARQUET format and ensure that all columns have the right schema (Almost). How would you do this?**

In [0]:
# We will use inferSchema = True while reading the data in that way spark will assign the best possible schema for the data

df = spark.read.format('parquet').option('inferSchema', True).load('source_path_location')

**15. You are reading a CSV file and need to handle corrupt records gracefully by skipping them. How would you configure this in PySpark?**

In [0]:
# We need use ".option('mode','DROP_MALFORMED')" while reading the data in that way all the corrupted data will be Droped

df = spark.read.format('csv').option('mode','DROP_MALFORMED').load('source_path_location')

# -Diff between RDDs, DATAFRAME and DATASET

**-RDD (Resilient Distributed Dataset):** This is the lowest-level data structure in Spark.
It consists of a distributed collection of objects without a predefined schema or structure. Because it lacks this structure, Spark cannot perform query optimization, making it generally slower to work with compared to higher-level abstractions.

**DataFrame:** This abstraction was introduced to address the limitations of RDDs. It provides a tabular structure with a schema, which enables Spark to perform query planning and optimizations. DataFrames are very popular and widely used because they are easy to use and compatible with various languages like Python (PySpark) and Scala.

**Dataset:** This structure combines the benefits of both RDDs and DataFrames. It provides a tabular structure with a schema (like a DataFrame) but adds type safety and support for functional programming.

# What is Query Optimization

**-Query Optimization** is the intelligent process Spark uses to transform raw code into an efficient execution plan before it runs on the cluster. Instead of simply executing code in the exact order written by the user, Spark optimizes the sequence of operations to ensure speed and resource efficiency.

**-Logical Plan:** The code is first analyzed to create a Logical Plan. The Catalyst Optimizer evaluates the various transformations (like join, select, or where clauses) and rearranges them. For example, if a select operation can reduce the volume of data before a join, Spark will perform the select first to minimize the workload.

**-Physical Plan:** Once the logical flow is optimized, Spark generates a Physical Plan. At this stage, it compares different strategies (such as sort merge joins vs. broadcast joins) against a cost model to select the most efficient approach for the specific hardware and data configuration.

**Execution as RDDs:** After the physical plan is finalized, it is converted into RDDs (Resilient Distributed Datasets). This allows the final, optimized workload to be distributed and processed in parallel across the worker nodes

# **What is Spark Session**

**-Unified Entry Point:** Before Spark 2.0, developers had to manage three different entry points: Spark Context, SQL Context, and Hive Context. Spark Session combines the functionalities of all three into one, simplifying development.

**Internal Management:** When using Spark Session, it internally manages the Spark Context, meaning developers do not need to create it explicitly.

**Recommendation:** While the older Spark Context can still be used for legacy applications,but Spark Session is the recommended approach for all development after Spark 2.0 due to its compatibility and ease of use.



# **Narrow Transformations and Wide transformations**


**-Narrow Transformations :** there is no data shuffling happens between machines.
For example : A filter operation (e.g., filtering IDs greater than 3) is a narrow transformation because each machine can process its local data independently without communicating with other nodes to return the final result

**-Wide transformations -** Data shuffling happens between machines.
Example : A groupBy operation is a classic wide transformation. If you want to group data by an ID column that is spread across different nodes, the system must shuffle the data to bring all records with the same ID into a single partition to calculate the aggregation correctly

# **COALESCE() and REPARTITION()**

**- COALESCE()** - It reduces the number of partitions. No Data Shufflings.
It simply merges existing partitions, which is much more efficient than repartitioning when you only need to decrease the partition count.
this is a "rule of thumb" for query optimization, as reading from fewer, larger partitions is often faster when dealing with big data.

**-REPARTITION()** - It increases the no. of partitions. Data shuffling happens.
The data is redistributed across the nodes in the cluster to create the new partition structure.
It is particularly useful when you have very large, uneven partitions that could lead to memory issues; repartition helps to evenly distribute the data across the available worker nodes.

# **CACHE() and PERSIST()**

-These are essential tools for managing data storage in PySpark when you need to reuse intermediate results to avoid recomputing them from scratch.

**Persist():** It allows you to define the storage level for your data (e.g., whether to store it in memory only, on disk only, or a combination of both serialized/deserialized).

Example: df.persist(StorageLevel.MEMORY_AND_DISK).

**Cache():** This is essentially a specialized convenience version of persist().

It is hardcoded to use the MEMORY_AND_DISK storage level by default.
Because this configuration is the most commonly used in practice, developers can simply call .cache() instead of typing out the full persist arguments 

In [0]:
from pyspark.storagelevel import StorageLevel

data = [("A", 1), ("B", 2), ("C", 3)]
columns = ["name", "value"]
df = spark.createDataFrame(data, columns)

# Using cache() - stores DataFrame in memory and disk by default
df.cache()
display(df)

# Using persist() - allows specifying storage level
df.persist(StorageLevel.MEMORY_AND_DISK)
display(df)

# **Importance of PARTITIONS in PYSPARK**

Partitions in PySpark as a fundamental mechanism that enables parallelism within the framework.

**Parallel Processing: **Partitioning involves breaking a massive dataset into smaller, manageable pieces (or chunks). This allows Spark to distribute the data across multiple machines or nodes in a cluster to be processed simultaneously.

This approach is known as Massive Parallel Processing (MPP). By splitting the workload, Spark can process huge datasets significantly faster than a single-node system.

**22. You have a dataset containing the names of employees and their departments. You need to find the department with the most employees.**

In [0]:
data = [("Alice", "HR"), ("Bob", "Finance"), ("Charlie", "HR"), ("David", "Engineering"), ("Eve", "Finance")]
columns = ["employee_name", "department"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
df = df.groupBy("department").agg(count("employee_name").alias('no_emp')).orderBy(col('no_emp').desc()).limit(1)

display(df)


**23. While processing sales data, you need to classify each transaction as either 'High' or 'Low' based on its amount. How would you achieve this using a when condition**

In [0]:
data = [("product1", 100), ("product2", 300), ("product3", 50)]
columns = ["product_id", "sales"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
df = df.withColumn('flag',when(col("sales")>50,"High").otherwise("low")).display()

**24. While analyzing a large dataset, you need to create a new column that holds a timestamp of when the record was processed. How would you implement this and what can be the best USE CASE?**

In [0]:
data = [("product1", 100), ("product2", 200), ("product3", 300)]
columns = ["product_id", "sales"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
df = df.withColumn('timestamp',current_timestamp()).display()

**25. You need to register this PySpark DataFrame as a temporary SQL object and run a query on it. How would you achieve this?**

In [0]:
data = [("product1", 100), ("product2", 200), ("product3", 300)]
columns = ["product_id", "sales"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
#First we need to create a Temp View

df.createOrReplaceTempView("temp_Sql_view")

spark.sql("Select * from temp_Sql_view").display()

In [0]:
%sql
Select * from temp_Sql_view

**26. You need to register this PySpark DataFrame as a temporary SQL object and run a query on it (FROM DIFFERENT NOTEBOOKS AS WELL)?**

We need to use df.createOrReplaceGlobalTempView('Give_Any_Global_name')

Then we need to use global_temp.Give_Any_Global_name while using SQL

like SELECT * from global_temp.Give_Any_Global_name

**27. You need to query data from a PySpark DataFrame using SQL, but the data includes a nested structure. How would you flatten the data for easier querying?**

In [0]:
data = [("product1", {"price": 100, "quantity": 2}), 
        ("product2", {"price": 200, "quantity": 3})]
columns = ["product_id", "product_info"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
df = df.select("product_id","product_info.price","product_info.quantity").display()

**28. You are ingesting data from an external API in JSON format where the schema is inconsistent. How would you handle this situation to ensure a robust pipeline?**

In [0]:
# By using mergeSchema option, we can merge the schema of the two DataFrames
df = spark.read.format('json').option('mergeSchema', True)

**29. While reading data from Parquet, you need to optimize performance by partitioning the data based on a column. How would you implement this?**

In [0]:
#We need to use partitionBY

df = spark.read.format('parquet').option('mode','append')\
    .partitionBy("<any_column>").load('path')

**30. You are working with a large dataset in Parquet format and need to ensure that the data is written in an optimized manner with proper compression. How would you accomplish this?**

In [0]:
#use Snappy Compression

df = spark.read.format('parquet').option("compression", "snappy").load('path')

**31. Your company uses a large-scale data pipeline that reads from Delta tables and processes data using complex aggregations. However, performance is becoming an issue due to the growing dataset size. How would you optimize the performance of the pipeline?**

By Using two primary optimization techniques: **OPTIMIZE** and **Z-ORDER BY**.

### **Key Optimization Strategies:**

*   **The OPTIMIZE Command:** This command is used to improve the layout of the data on disk. It compacts small files into larger, more manageable ones, which significantly improves read performance and efficiency.

*   **Z-ORDER BY:** This is an advanced technique that clusters related information together in the same set of files. By using `Z-ORDER BY` on a column frequently used in filters or joins (e.g., `order_date`), you enable *data skipping*. 
    *   **How it works:** *Delta Lake* maintains **column statistics** for the first 32 columns. When you `Z-ORDER` by a column, the engine can quickly identify and skip files that do not contain the data relevant to your query, thereby reducing the amount of data scanned and boosting speed.

*   **Pro-tip:** Always ensure that your `Z-ORDER BY` column is among the first 32 columns of your table to leverage the automatic statistics collection effectively (2:46:05).

In [0]:
%sql
-- Example: OPTIMIZE with ZORDER BY for improving query performance on frequently filtered columns
OPTIMIZE my_table
ZORDER BY (user_id, event_date);

-- Example: OPTIMIZE with liquid clustering for adaptive clustering based on data distribution
CREATE TABLE catalog.schema.table_name (
  column1 TYPE,
  column2 TYPE,
  ...
)
CLUSTER BY (column1, column2);

-- Example: Writing data with optimization options (e.g., partitioning, file size, and schema evolution)
CREATE TABLE my_table
USING DELTA
PARTITIONED BY (event_date)
LOCATION '/mnt/data/my_table';

-- Example: Writing data with optimized file sizes
INSERT INTO my_table
SELECT * FROM staging_table
OPTIONS ('maxFilesPerTrigger' = '10', 'mergeSchema' = 'true');

# What are Broadcast Variables ?

In *PySpark*, **broadcast variables** are a mechanism for sending a read-only variable to each worker node in a cluster, rather than sending a copy of it with every single task.

### **Why use Broadcast Variables?**
* **Efficiency:** They help eliminate significant networking overhead. Instead of serializing and sending large reference data (like a lookup table, list of country codes, or a small reference dataset) to every executor repeatedly, the data is sent to each node only once.
* **Optimized Joins:** They are particularly useful for optimizing **joins** when one of your datasets is small enough to fit into the memory of the worker nodes. By broadcasting the smaller dataset, you avoid expensive data shuffling across the network.

**Key Takeaway:** Think of them as a way to share read-only data across your distributed system efficiently, preventing the need for the driver to transmit the same information multiple times.

In [0]:
# Definition of Broadcast Variables:
# In PySpark, broadcast variables are used to efficiently share a read-only variable (such as a lookup table or reference data)
# across all worker nodes in a cluster. Instead of sending the variable with every task, Spark sends it once to each node,
# reducing network overhead and improving performance.

# How it works:
# 1. The variable is broadcasted from the driver to all executors.
# 2. Each executor stores the broadcasted variable in memory.
# 3. Tasks running on the executor can access the broadcasted variable locally, avoiding repeated data transfer.

# Example:
country_codes = {"US": "United States", "IN": "India", "UK": "United Kingdom"}
broadcast_var = spark.sparkContext.broadcast(country_codes)

# Use broadcast_var.value to access the broadcasted data in your transformations or UDFs.

In [0]:
# Sample code to demonstrate Broadcast Variables in PySpark

# Reference data to broadcast (e.g., country codes)
country_codes = {"US": "United States", "IN": "India", "UK": "United Kingdom"}
broadcast_var = spark.sparkContext.broadcast(country_codes)

# Sample DataFrame with country abbreviations
data = [("Alice", "US"), ("Bob", "IN"), ("Charlie", "UK")]
columns = ["name", "country_code"]
df = spark.createDataFrame(data, columns)

# Use the broadcast variable in a UDF to map country codes to country names
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def get_country_name(code):
    return broadcast_var.value.get(code, "Unknown")

country_name_udf = udf(get_country_name, StringType())

df = df.withColumn("country_name", country_name_udf(df["country_code"]))
display(df)

# **Diff b/w df.show() and df.collect()**

**df.show():** This method is primarily used to get a preview or an overview of your data. It returns a limited number of rows (defaulting to a small sample) in a formatted table, making it ideal for checking your data during development without overloading the system.

![image_1785233323106.png](./image_1785233323106.png "image_1785233323106.png")


**df.collect():** This method returns all rows of the data frame as a list of objects to the driver node. The creator warns to use this cautiously; because it attempts to bring the entire dataset into the driver's memory, it can lead to a "driver out of memory" error if the dataset is large.

![image_1785233379994.png](./image_1785233379994.png "image_1785233379994.png")

# **What is LAZY EVALUATION**

In *PySpark*, **Lazy Evaluation** is a core design principle where transformations on a DataFrame are not executed immediately when they are defined.

### **How it Works:**
* **Recording Transformations:** When you perform operations like `filter()`, `select()`, or `sort()`, Spark does not actually process any data. Instead, it records these steps in a **logical execution plan**.
* **The Trigger (Action):** The actual computation only begins when an **Action** is triggered (e.g., `df.show()`, `df.collect()`, `df.count()`, or writing data to a file). 
* **Optimization:** Because Spark knows the full sequence of operations before executing them, the **Catalyst Optimizer** can analyze the entire plan to perform optimizations—such as filtering data earlier or reordering steps—to ensure the query runs as efficiently as possible.

**Key Takeaway:** Lazy evaluation allows *PySpark* to build a highly optimized workflow, ensuring that the system only processes exactly what is needed, which significantly enhances performance when dealing with large-scale distributed data.

# Advantages of Delta Lake

1.  **ACID Transactions:** *Delta Lake* ensures **atomicity, consistency, isolation, and durability** (ACID) by using a *Delta log*. This guarantees that data operations are reliable and maintain integrity, even in concurrent environments.

2.  **Schema Enforcement and Evolution:** Unlike traditional formats that primarily use "schema-on-read," *Delta Lake* applies **schema-on-write**. It checks the target schema against the source, preventing data corruption by throwing errors on mismatch, while also allowing for controlled **schema evolution**.

3.  **Optimization Techniques:** *Delta Lake* provides powerful tools to optimize large datasets, such as the `OPTIMIZE` command and `Z-ORDER BY`, which improve query performance and efficiency through **data skipping**.

In [0]:
# Sample code to demonstrate Schema Enforcement and Evolution in Delta tables

data1 = [(1, "Alice", 5000)]
columns1 = ["ID", "Name", "Salary"]

df1 = spark.createDataFrame(data1, columns1)
df1.write.format("delta").mode("overwrite").save("/tmp/delta/schema_demo")

# Schema Enforcement: Attempting to write with a mismatched schema will fail
data2 = [(2, "Bob", "6000")]  # Salary as string instead of integer
columns2 = ["ID", "Name", "Salary"]
df2 = spark.createDataFrame(data2, columns2)
try:
    df2.write.format("delta").mode("append").save("/tmp/delta/schema_demo")
except Exception as e:
    print("Schema Enforcement Error:", e)

# Schema Evolution: Adding a new column and enabling schema evolution
data3 = [(3, "Charlie", 7000, "charlie@example.com")]
columns3 = ["ID", "Name", "Salary", "Email"]
df3 = spark.createDataFrame(data3, columns3)
df3.write.format("delta").option("mergeSchema", "true").mode("append").save("/tmp/delta/schema_demo")

# Display the evolved schema
df_final = spark.read.format("delta").load("/tmp/delta/schema_demo")
display(df_final)

# **what happens when a PySpark job runs out of memory ?**

When a *PySpark* job encounters memory issues, it is commonly referred to as an **Out of Memory (OOM)** error.

*   **Driver Out of Memory:** This occurs when the *Driver node* attempts to collect more data than it can handle. A common cause is using `df.collect()` on a very large dataset, which forces the driver to pull all rows into its local memory.

*   **Executor Out of Memory:** This happens when an *Executor node* runs out of memory while processing tasks. This is frequently caused by **data skewness**, where one executor is assigned a disproportionately large partition of data that exceeds its memory capacity, often during expensive join operations.

### **Mitigation Strategies:**
*   **Increase Resources:** Increase the memory allocation for either the *Driver* or the *Executors* (2:59:39).
*   **Adaptive Query Execution (AQE):** Modern Spark uses *AQE* to automatically detect and handle skewness at runtime, reducing the likelihood of OOM errors (3:01:11 - 3:04:14).
*   **Salting:** An older manual technique involving adding a random suffix (a "salt") to keys to distribute skewed data more evenly across nodes (2:59:58 - 3:00:26).
*   **Spilling:** If memory is insufficient, Spark may "spill" data to the **disk** to continue processing. While this prevents a crash, it significantly degrades performance (3:11:06 - 3:12:43).

# **What is AQE**

**AQE** (Adaptive Query Execution) is a powerful optimization technique in *PySpark* that improves query execution at runtime. 

Unlike traditional static optimization that relies on plans created before the job starts, *AQE* re-optimizes the query plan as data is being processed, making it highly effective for handling dynamic data conditions. 

**Key features of AQE include:**

* **Dynamic Partition Pruning:** Optimizes filter operations by pruning unnecessary partitions based on join conditions at runtime.

* **Join Strategy Optimization:** Dynamically chooses the most efficient join method (e.g., switching to a broadcast join) based on actual data sizes observed during execution.

* **Dynamically Optimizing Skewed Joins:** Automatically detects and breaks down skewed partitions (where one partition contains significantly more data than others) into smaller pieces to balance the workload across executors, preventing performance bottlenecks and potential memory errors.

In modern *PySpark* versions (3.0 and above), **AQE is auto-enabled**, meaning you generally do not need to manually configure it to benefit from these optimizations.

# **"how would you handle skewed data in pySpark" ?**

**data skewness** occurs when data is unevenly distributed across partitions or executors, often causing performance bottlenecks or "Executor Out of Memory" errors. 

**Two main ways to handle skewed data in *PySpark*:**

* **Adaptive Query Execution (AQE):** This is the modern, preferred approach. Since *Spark 3.0*, *AQE* is auto-enabled, meaning *PySpark* can automatically detect skew at runtime and break down skewed partitions into smaller, more manageable pieces to balance the workload across executors.

* **Salting:** This is an older, manual technique used if *AQE* is not available or sufficient. It involves adding a random suffix (a "salt") to the joining key (e.g., changing a skewed key '1' into '1a', '1b', '1c'). This splits the heavily-frequented key into smaller, distinct groups, allowing the data to be spread across multiple executors instead of overwhelming just one.

In [0]:
#Sample code to show SALTING

from pyspark.sql.functions import col, concat, lit, rand, floor

# Sample skewed data
data = [(1, "A"), (1, "B"), (1, "C"), (2, "D"), (3, "E")]
df = spark.createDataFrame(data, ["key", "value"])

# Add salt to the skewed key
num_salts = 3
# floor(rand() * num_salts) generates a random integer between 0 and num_salts-1 for each row,
# which is used as a "salt" to distribute skewed keys across multiple partitions.

df_salted = df.withColumn("salt", floor(rand() * num_salts)) \
              .withColumn("salted_key", concat(col("key"), lit("_"), col("salt")))

display(df_salted)

# What is broadcast join ? when to use it ?

In *PySpark*, a **broadcast join** is an optimization technique used when joining a large dataset with a significantly smaller one. 

**How it works:**
* Instead of shuffling both datasets across the network—which is an expensive operation—Spark broadcasts (copies) the small dataset to all executor nodes in the cluster.
* Each executor then performs the join locally using its portion of the large dataset and the full copy of the small dataset stored in its memory.
* This eliminates the need for massive data movement across the network.

from pyspark.sql.functions import broadcast

**When to use it:**
* Use this approach when one of the dataframes is small enough to comfortably fit in the memory of the executors.
* While *Spark* automatically performs a broadcast join if a table falls below a certain size threshold (configured by the user), you can manually trigger it using the `broadcast()` function in your code to explicitly hint that a specific dataframe should be broadcast.

By keeping the small table locally on each worker node, you drastically reduce networking overhead and improve overall query performance.

In [0]:
#Sample to show Broadcast Join

from pyspark.sql.functions import broadcast

# Sample large and small DataFrames
large_data = [(1, "A"), (2, "B"), (3, "C")]
small_data = [(1, "Alpha"), (2, "Beta")]

large_df = spark.createDataFrame(large_data, ["id", "value"])
small_df = spark.createDataFrame(small_data, ["id", "desc"])

# Broadcast join
result = large_df.join(broadcast(small_df), "id")

display(result)

# **What is SPILL in Spark ?**

In *PySpark*, **spill** (or spilling to disk) is a mechanism that occurs when an **executor node** does not have enough memory to process intermediate data or transformations.

**Key takeaways about spilling:**
* **Why it happens:** It is essentially a safety fallback triggered when a task's memory requirements exceed the available RAM allocated to the executor.

* **The Process:** Instead of crashing the job with an "Out of Memory" (OOM) error, *Spark* offloads the overflow data to the **disk** to continue processing .

* **Performance Impact:** While spilling allows the job to complete rather than failing, it **significantly degrades performance** because reading from and writing to disk is much slower than processing data directly in memory.

* **Common Causes:** Spilling is often the result of data skewness (uneven data distribution) or expensive operations like complex joins that consume large amounts of memory.
* **How to Mitigate:** To prevent or reduce spilling, you can increase the allocated executor memory or implement techniques like **salting** or **AQE** to handle skewed data more efficiently.

# **Time Travel Feature**

The **Time Travel** feature in *Delta Lake* is a powerful tool for data versioning and recovery. It allows you to query historical versions of a data table or even restore the table to a previous state if data was accidentally deleted or modified.

**How it works:**

* **Viewing History:** You can inspect the changes made to a *Delta* table by running the command `DESCRIBE HISTORY table_name`. This provides a list of all versions (e.g., version 0, 1, 2, etc.) along with the activities performed on them.

* **Restoring Data:** If you find that a mistake (such as accidental deletion) occurred at a specific version, you can revert the table back to a known good state using the command `RESTORE TABLE table_name TO VERSION AS OF <version_number>`.

This feature is highly regarded in data engineering as a critical safety net, effectively allowing you to "travel back in time" to recover lost data without needing to perform complex restores from backups.

**43. You are processing sales data. Group by product categories and create a list of all product names in each category.**

In [0]:
data = [("Electronics", "Laptop"), ("Electronics", "Smartphone"), ("Furniture", "Chair"), ("Furniture", "Table")]
columns = ["category", "product"]
df = spark.createDataFrame(data, columns)
df.display()

In [0]:
df = df.groupBy('category').agg(collect_list("product").alias('product_list'))

display(df)

**44. You are analyzing orders. Group by customer IDs and list all unique product IDs each customer purchased.**

In [0]:
data = [(101, "P001"), (101, "P002"), (102, "P001"), (101, "P001")]
columns = ["customer_id", "product_id"]
df = spark.createDataFrame(data, columns)
df.display()

In [0]:
df = df.groupBy("customer_id").agg(collect_set("product_id")).display()

**45. For customer records, combine first and last names only if the email address exists.**

In [0]:
data = [("John", "Doe", "john.doe@example.com"), ("Jane", "Smith", None)]
columns = ["first_name", "last_name", "email"]
df = spark.createDataFrame(data, columns)
df.display()

In [0]:
df = df.withColumn('FullName',when(col("email").isNotNull(),concat_ws(" ","first_name","last_name")).otherwise(None))

display(df)

**46. You have a DataFrame containing customer IDs and a list of their purchased product IDs. Calculate the number of products each customer has purchased.**

In [0]:
data = [
    (1, ["prod1", "prod2", "prod3"]),
    (2, ["prod4"]),
    (3, ["prod5", "prod6"]),
]
myschema = "customer_id INT ,product_ids array<STRING>"

df = spark.createDataFrame(data, myschema)
df.display()

In [0]:
df = df.withColumn('no_products',size("product_ids")).display()

**Another Method using EXPLODE function**

In [0]:
df = df.select("customer_id", explode("product_ids").alias("product_id"))

display(df)

In [0]:
df= df.groupBy('customer_id').agg(count("product_id").alias('no._products')).display()

**47. You have employee IDs of varying lengths. Ensure all IDs are 6 characters long by padding with leading zeroes.**

In [0]:
data = [
    ("1",),
    ("123",),
    ("4567",),
]
schema = ["employee_id"]

df = spark.createDataFrame(data, schema)
df.display()

In [0]:
df = df.withColumn("emp_id_2",lpad(col("employee_id"),6,"0")).display()

**48. You need to validate phone numbers by checking if they start with "91"**

In [0]:
data = [
    ("911234567890",),
    ("811234567890",),
    ("912345678901",),
]
columns = ["phone_number"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

df = df.withColumn('ph_2',substring("phone_number",1,2)).filter(col('ph_2')=='91')

display(df)

In [0]:
df = df.filter(substring("phone_number",1,2)=="91").display()

**49. You have a dataset with courses taken by students. Calculate the average number of courses per student.**

In [0]:
data = [
    (1, ["Math", "Science"]),
    (2, ["History"]),
    (3, ["Art", "PE", "Biology"]),
]
schema = ["student_id", "courses"]

df = spark.createDataFrame(data, schema)
df.display()

In [0]:
df = df.withColumn("no_courses",size(col("courses")))

#display(df)

df = df.groupBy().agg(avg("no_courses").alias("avg_courses")).display()

**50. You have a dataset with primary and secondary contact numbers. Use the primary number if available; otherwise, use the secondary number.**

In [0]:
data = [
    (None, "1234567890"),
    ("9876543210", None),
    ("7894561230", "4567891230"),
]
schema = ["primary_contact", "secondary_contact"]

df = spark.createDataFrame(data, schema)
df.display()

In [0]:
df = df.withColumn('contact',coalesce(col("primary_contact"),col("secondary_contact"))).display()

**51. You are categorizing product codes based on their lengths. If the length is 5, label it as "Standard"; otherwise, label it as "Custom".**

In [0]:
data = [
    ("prod1",),
    ("prd234",),
    ("pr9876",),
]
schema = ["product_code"]

df = spark.createDataFrame(data, schema)
df.display()

In [0]:
df = df.withColumn("length",length(col('product_code')))

df = df.withColumn('test',when(col('length')==5,"Standard").otherwise("custom")).display()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

data =[(101,'10:00'),(101,'10:10'),(101,'10:25'),(101,'11:10'),(101,'11:20'),(102,'09:00'),(102,'09:45')]
columns = ["user_id","click_time"]

df = spark.createDataFrame(data,columns)

display(df)

df.createOrReplaceTempView("user_session_2")

In [0]:
%sql
Select *,case 
when gap is NULL then 0
when gap > 30 then 1
else 0 end as session_id from
(SELECT user_id,click_time,prev_click,timediff(minute,prev_click,click_time) as gap 
from 
(SELECT 
        user_id,
        click_time::timestamp,lag(click_time::timestamp) OVER(PARTITION BY user_id ORDER BY click_time::timestamp) as prev_click
    FROM user_session_2
    )p)k

**Session ID Assignment Problem:**

Assign a unique session ID to every click where a new session starts when time difference between consecutive clicks for the same user is > 30 minutes.

In [0]:
%sql
WITH click_with_prev AS (
    SELECT 
        user_id,
        click_time,
        LAG(click_time, 1) OVER (PARTITION BY user_id ORDER BY click_time) AS prev_click_time
    FROM user_session_2
),
session_markers AS (
    SELECT 
        user_id,
        click_time,
        prev_click_time,
        -- Calculate time difference in minutes
        CASE 
            WHEN prev_click_time IS NULL THEN NULL
            ELSE (UNIX_TIMESTAMP(click_time) - UNIX_TIMESTAMP(prev_click_time)) / 60
        END AS time_diff_minutes,
        -- Mark new session: first click for user OR time diff > 30 minutes
        CASE 
            WHEN prev_click_time IS NULL THEN 1  -- First click for this user
            WHEN (UNIX_TIMESTAMP(click_time) - UNIX_TIMESTAMP(prev_click_time)) / 60 > 30 THEN 1  -- Gap > 30 min
            ELSE 0  -- Same session
        END AS is_new_session
    FROM click_with_prev
)
SELECT 
    user_id,
    click_time,
    time_diff_minutes,
    is_new_session,
    -- Create cumulative session ID by summing the session markers
    CONCAT(user_id, '_', SUM(is_new_session) OVER (PARTITION BY user_id ORDER BY click_time)) AS session_id
FROM session_markers
ORDER BY user_id, click_time

**Clean Solution (Final Output):**

In [0]:
%sql
WITH session_markers AS (
    SELECT 
        user_id,
        click_time,
        -- Mark new session: first click for user OR time diff > 30 minutes
        CASE 
            WHEN LAG(click_time, 1) OVER (PARTITION BY user_id ORDER BY click_time) IS NULL THEN 1
            WHEN (UNIX_TIMESTAMP(click_time) - UNIX_TIMESTAMP(LAG(click_time, 1) OVER (PARTITION BY user_id ORDER BY click_time))) / 60 > 30 THEN 1
            ELSE 0
        END AS is_new_session
    FROM user_session_2
)
SELECT 
    user_id,
    click_time,
    CONCAT(user_id, '_', SUM(is_new_session) OVER (PARTITION BY user_id ORDER BY click_time)) AS session_id
FROM session_markers
ORDER BY user_id, click_time

**Explanation:**

**Approach:**
1. Use `LAG()` window function to get the previous click_time for each user (partitioned by user_id, ordered by click_time)
2. Calculate time difference in minutes between current and previous click
3. Mark new session starts:
   - First click for a user (prev_click_time is NULL) → `is_new_session = 1`
   - Time difference > 30 minutes → `is_new_session = 1`
   - Otherwise → `is_new_session = 0`
4. Use `SUM(is_new_session) OVER (PARTITION BY user_id ORDER BY click_time)` to create cumulative session IDs

**Results:**
- **User 101:**
  - Session 101_1: clicks at 10:00, 10:10, 10:25 (gaps of 10 and 15 minutes)
  - Session 101_2: clicks at 11:10, 11:20 (started after 45-minute gap)

- **User 102:**
  - Session 102_1: click at 09:00
  - Session 102_2: click at 09:45 (started after 45-minute gap)

# Scenario

**Suppose you have a file in which i need to separate valid and invalid columns.
Valid columns are those in which the blank % is less than 20. And we also need to have a list of invalid columns and then save the dataframe in parquet format selecting only valid columns. How can this be done**

In [0]:
# Create a sample table with varying blank percentages
data = [
    (1, "Alice", 28, "Engineering", "alice@example.com", "New York", 85000, "Senior"),
    (2, "Bob", 35, "Marketing", None, "Los Angeles", 72000, None),
    (3, "Charlie", None, "Sales", "charlie@example.com", "Chicago", None, "Junior"),
    (4, "Diana", 42, "Engineering", "diana@example.com", None, 95000, "Lead"),
    (5, "Eve", 29, None, "eve@example.com", "Boston", 68000, "Mid"),
    (6, "Frank", 31, "Sales", None, "Seattle", None, None),
    (7, "Grace", 38, "HR", "grace@example.com", "Austin", 78000, "Senior"),
    (8, "Henry", None, "Engineering", "henry@example.com", "Denver", 82000, None),
    (9, "Iris", 26, "Marketing", None, None, None, "Junior"),
    (10, "Jack", 33, "Sales", "jack@example.com", "Miami", 71000, "Mid"),
]

columns = ["employee_id", "name", "age", "department", "email", "city", "salary", "level"]

df = spark.createDataFrame(data, columns)
df.display()

In [0]:
# VIJAY SOLUTION


from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *


val_col = []
invalid_col = []
log_table = []

total_rows = df.count()

for i in df.columns:
    # Calculate blank percentage for each column
    blank_count = df.filter(col(i).isNull()).count()
    blank_percentage = (blank_count / total_rows) * 100
    
    # Separate valid and invalid columns based on 20% threshold
    if blank_percentage < 20:
        val_col.append(i)
        status = "Valid"
    else:
        invalid_col.append(i)
        status = "Invalid"

    log_table.append((i,blank_count,blank_percentage,status))

log_schema = ["col_name","blank_count","blank_percentage","status"]
df_log = spark.createDataFrame(log_table,log_schema)

display(df_log)



print(val_col)

# Select only valid columns and save as parquet
df_valid = df.select(val_col)
df_invalid = df.select(invalid_col)
display(df_valid)
display(df_invalid)

# df_valid.write.format('parquet').mode('append').save('/processed')

# df_invalid.write.format('parquet').mode('append').save('/in_valid_records')

#print(f"\nDataFrame with valid columns saved to parquet format")




In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime
from delta.tables import DeltaTable
import builtins

# Generate unique batch_id for this run (date-based for daily idempotency)
batch_id = datetime.now().strftime('%Y-%m-%d')  # Use date as batch_id for daily runs
process_timestamp = datetime.now()

val_col = []
invalid_col = []
log_table = []

total_rows = df.count()

for i in df.columns:
    blank_count = df.filter(col(i).isNull()).count()
    blank_percentage = (blank_count / total_rows) * 100
    
    if blank_percentage < 20:
        val_col.append(i)
        status = "Valid"
    else:
        invalid_col.append(i)
        status = "Invalid"
    
    # Include batch_id in log entry
    log_table.append((
        batch_id,
        i,
        blank_count,
        builtins.round(blank_percentage, 2),
        status,
        process_timestamp
    ))

# Create log DataFrame with batch_id
log_schema = ["batch_id", "col_name", "blank_count", "blank_percentage", "status", "process_timestamp"]
df_log = spark.createDataFrame(log_table, log_schema)

print(f"Batch ID: {batch_id}")
print(f"Valid columns: {val_col}")
print(f"Invalid columns: {invalid_col}")

display(df_log)

# Idempotent write using MERGE
# Use a new table name to avoid schema conflicts with existing table
target_table = "pyspark_cata_vijay.source.column_validation_log_idempotent"

try:
    # Check if table exists
    target_delta = DeltaTable.forName(spark, target_table)
    
    # MERGE: Update if (batch_id, col_name) exists, otherwise insert
    target_delta.alias('target').merge(
        df_log.alias('source'),
        'target.batch_id = source.batch_id AND target.col_name = source.col_name'
    ).whenMatchedUpdateAll(
    ).whenNotMatchedInsertAll(
    ).execute()
    
    print(f"✓ Log table updated idempotently (MERGE completed)")
    print(f"  If you run this cell again with the same batch_id, it will UPDATE existing records, not duplicate them.")
    
except Exception as e:
        df_log.write.format('delta').mode('overwrite').saveAsTable(target_table)
        

# Select and display valid/invalid columns
df_valid = df.select(val_col)
df_invalid = df.select(invalid_col) if invalid_col else None

print("\n=== Valid Columns Data ===")
display(df_valid)

if df_invalid:
    print("\n=== Invalid Columns Data ===")
    display(df_invalid)

# Idempotency Implementation Patterns

## What is Idempotency?
Idempotency means running the same operation multiple times produces the same result as running it once - no duplicate data, no side effects.

---

## Approach 1: MERGE (UPSERT) Pattern - Recommended for Most Cases

**How it works:**
* Uses Delta MERGE operation to upsert based on a composite key (batch_id + col_name)
* If record exists: UPDATE it
* If record does not exist: INSERT it

**When to use:**
* When you want to update existing records if they have changed
* When you need row-level idempotency (same batch_id can update metrics)
* Most flexible - handles both inserts and updates

**Pros:**
* Handles schema evolution
* Most production-ready pattern
* Can update changed values without duplicating

**Cons:**
* Slightly more complex code
* Requires Delta Lake (not plain Parquet)

---

## Approach 2: Check-Before-Insert Pattern

**How it works:**
* Query the target table to check if batch_id already exists
* If exists: SKIP insert
* If not exists: INSERT

**When to use:**
* When you never want to update existing records (append-only log)
* When you want explicit control over duplicate prevention
* When you need to log when duplicates are skipped

**Pros:**
* Simple and explicit
* Clear logging of skipped runs
* No overwrites - preserves original data

**Cons:**
* Requires full table scan to check existence (slower for large tables)
* Does not handle updates - only prevents duplicates

---

## Approach 3: Partition Overwrite Pattern - Best for Daily/Time-Based Jobs

**How it works:**
* Partition data by run_date (or another time dimension)
* Use replaceWhere to overwrite ONLY today's partition
* Other partitions remain untouched

**When to use:**
* Daily/hourly batch jobs that re-run for the same time period
* When you want to replace all data for a specific date (not individual records)
* When late-arriving data requires reprocessing a specific date

**Pros:**
* Fastest - no row-level lookups or merges
* Simplest for time-series data
* Naturally idempotent for date-based processing

**Cons:**
* Overwrites entire partition (cannot update individual records)
* Requires data to be naturally partitionable (dates, regions, etc.)

---

## Quick Decision Guide

| Requirement | Best Approach |
|-------------|---------------|
| Daily batch job, reprocess same day | Approach 3 (Partition Overwrite) |
| Need to update changed records | Approach 1 (MERGE) |
| Append-only log, never update | Approach 2 (Check-Before-Insert) |
| Hourly job within same day | Approach 1 (MERGE) with timestamp batch_id |
| Production ETL pipeline | Approach 1 (MERGE) or Approach 3 |
| Simple one-time migration | Approach 2 |

---

## Interview Talking Points

1. **Why idempotency matters:**
   * Jobs fail and need to be retried
   * Orchestration tools may trigger the same task twice
   * Prevents data duplication and maintains data quality

2. **Delta Lake advantages:**
   * ACID transactions ensure consistency
   * MERGE operation is atomic
   * Time travel lets you audit changes

3. **Performance considerations:**
   * MERGE is fast with proper partitioning and Z-ordering
   * Check-before-insert can be slow on large tables (full scan)
   * Partition overwrite is fastest for bulk updates

4. **Real-world scenario:**
   * Our daily sales aggregation job runs at 2 AM. If it fails at 90% and retries at 2:30 AM, we use partition overwrite on date = today() so the retry replaces partial data with complete results.

In [0]:
%sql
-- Query the log table to see the results
-- Notice: batch_id = '2026-09-10' (today's date)
-- If you run the MERGE cell multiple times today, you'll see the SAME 8 rows (not duplicates)
-- The process_timestamp will update to the latest run time

SELECT 
    batch_id,
    col_name,
    blank_percentage,
    status,
    process_timestamp
FROM pyspark_cata_vijay.source.column_validation_log_idempotent
ORDER BY batch_id DESC, col_name

In [0]:
%sql
-- Demonstrate idempotency:
-- This query shows the total row count in the log table
-- 
-- WITHOUT idempotency: Running the validation 3 times would give 24 rows (8 columns × 3 runs)
-- WITH idempotency (MERGE): Running 3 times still gives 8 rows (batch_id + col_name is unique)
--
-- Try it: Run the MERGE cell (Approach 1) multiple times, then run this query
-- You'll always see 8 rows for today's batch_id (not 16, 24, 32...)

SELECT 
    COUNT(*) as total_rows,
    COUNT(DISTINCT batch_id) as unique_batches,
    COUNT(DISTINCT col_name) as unique_columns
FROM pyspark_cata_vijay.source.column_validation_log_idempotent

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime
from delta.tables import DeltaTable
import builtins

# Generate unique batch_id for this run
batch_id = datetime.now().strftime('%Y-%m-%d')
process_timestamp = datetime.now()

val_col = []
invalid_col = []
log_table = []

total_rows = df.count()

for i in df.columns:
    blank_count = df.filter(col(i).isNull()).count()
    blank_percentage = (blank_count / total_rows) * 100
    
    if blank_percentage < 20:
        val_col.append(i)
        status = "Valid"
    else:
        invalid_col.append(i)
        status = "Invalid"
    
    log_table.append((
        batch_id,
        i,
        blank_count,
        builtins.round(blank_percentage, 2),
        status,
        process_timestamp
    ))

log_schema = ["batch_id", "col_name", "blank_count", "blank_percentage", "status", "process_timestamp"]
df_log = spark.createDataFrame(log_table, log_schema)

print(f"Batch ID: {batch_id}")
print(f"Valid columns: {val_col}")
print(f"Invalid columns: {invalid_col}")

display(df_log)

target_table = "pyspark_cata_vijay.source.column_validation_log_partial_update"

try:
    target_delta = DeltaTable.forName(spark, target_table)
    
    # MERGE with SELECTIVE UPDATE: Update only blank_count and blank_percentage
    # Other columns (batch_id, col_name, status, process_timestamp) remain unchanged
    target_delta.alias('target').merge(
        df_log.alias('source'),
        'target.batch_id = source.batch_id AND target.col_name = source.col_name'
    ).whenMatchedUpdate(
        set = {
            "blank_count": "source.blank_count",
            "blank_percentage": "source.blank_percentage"
        }
    ).whenNotMatchedInsertAll(
    ).execute()
    
    print(f"✓ Log table updated (ONLY blank_count and blank_percentage updated)")
    print(f"  Note: status and process_timestamp in existing records remain unchanged")
    
except Exception as e:
    error_msg = str(e).lower()
    if "not found" in error_msg or "not a delta table" in error_msg or "does not exist" in error_msg:
        df_log.write.format('delta').mode('overwrite').saveAsTable(target_table)
        print(f"✓ Log table created: {target_table}")
        print(f"  Run this cell again to test partial update")
    else:
        raise e

# Display result
print("\n=== Log Table (Partial Update Demo) ===")
spark.table(target_table).orderBy("batch_id", "col_name").display()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime
from delta.tables import DeltaTable
import builtins

batch_id = datetime.now().strftime('%Y-%m-%d')
process_timestamp = datetime.now()

val_col = []
invalid_col = []
log_table = []

total_rows = df.count()

for i in df.columns:
    blank_count = df.filter(col(i).isNull()).count()
    blank_percentage = (blank_count / total_rows) * 100
    
    if blank_percentage < 20:
        val_col.append(i)
        status = "Valid"
    else:
        invalid_col.append(i)
        status = "Invalid"
    
    log_table.append((
        batch_id,
        i,
        blank_count,
        builtins.round(blank_percentage, 2),
        status,
        process_timestamp
    ))

log_schema = ["batch_id", "col_name", "blank_count", "blank_percentage", "status", "process_timestamp"]
df_log = spark.createDataFrame(log_table, log_schema)

target_table = "pyspark_cata_vijay.source.column_validation_log_conditional"

try:
    target_delta = DeltaTable.forName(spark, target_table)
    
    # ADVANCED: Multiple MERGE conditions with different update strategies
    target_delta.alias('target').merge(
        df_log.alias('source'),
        'target.batch_id = source.batch_id AND target.col_name = source.col_name'
    ).whenMatchedUpdate(
        # Update ONLY if blank_percentage has changed by more than 5%
        condition = "abs(source.blank_percentage - target.blank_percentage) > 5",
        set = {
            "blank_count": "source.blank_count",
            "blank_percentage": "source.blank_percentage",
            "status": "source.status",
            "process_timestamp": "source.process_timestamp"
        }
    ).whenNotMatchedInsertAll(
    ).execute()
    
    print(f"✓ Conditional update completed")
    print(f"  Only updated records where blank_percentage changed by more than 5%")
    
except Exception as e:
    error_msg = str(e).lower()
    if "not found" in error_msg or "not a delta table" in error_msg or "does not exist" in error_msg:
        df_log.write.format('delta').mode('overwrite').saveAsTable(target_table)
        print(f"✓ Log table created: {target_table}")
    else:
        raise e

print("\n=== Log Table (Conditional Update Demo) ===")
spark.table(target_table).orderBy("batch_id", "col_name").display()

# Delta Lake MERGE Update Patterns

## 1. Update ALL Columns (Default)

```python
target_delta.alias('target').merge(
    df_log.alias('source'),
    'target.batch_id = source.batch_id AND target.col_name = source.col_name'
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()
```

**What it does:** Updates ALL columns from source to target when match found

**Use case:** When you want to refresh all fields with latest values

---

## 2. Update ONLY Specific Columns

```python
target_delta.alias('target').merge(
    df_log.alias('source'),
    'target.batch_id = source.batch_id AND target.col_name = source.col_name'
).whenMatchedUpdate(
    set = {
        "blank_count": "source.blank_count",
        "blank_percentage": "source.blank_percentage"
    }
).whenNotMatchedInsertAll(
).execute()
```

**What it does:** Updates ONLY blank_count and blank_percentage. Other columns (status, process_timestamp) remain unchanged

**Use case:** 
* When you want to update metrics but preserve original metadata
* When you want to track first-seen timestamp but update current values
* When certain columns should be immutable after creation

**Example scenario:** Original record has `process_timestamp = 2026-09-10 03:00:00`. When you run the merge again at 03:30:00, the blank_count and blank_percentage update, but process_timestamp stays at 03:00:00 (the first time it was inserted)

---

## 3. Update with CONDITION

```python
target_delta.alias('target').merge(
    df_log.alias('source'),
    'target.batch_id = source.batch_id AND target.col_name = source.col_name'
).whenMatchedUpdate(
    condition = "source.blank_percentage != target.blank_percentage",
    set = {
        "blank_count": "source.blank_count",
        "blank_percentage": "source.blank_percentage",
        "process_timestamp": "source.process_timestamp"
    }
).whenNotMatchedInsertAll(
).execute()
```

**What it does:** Updates ONLY if the condition is met (value has changed)

**Use case:**
* Avoid unnecessary writes when data has not changed
* Track only meaningful changes
* Improve performance by reducing write volume

**Common conditions:**
* `source.blank_percentage != target.blank_percentage` - update only if value changed
* `abs(source.blank_percentage - target.blank_percentage) > 5` - update only if changed by more than 5%
* `source.status != target.status` - update only if status changed

---

## 4. Multiple whenMatchedUpdate Clauses

```python
target_delta.alias('target').merge(
    df_log.alias('source'),
    'target.batch_id = source.batch_id AND target.col_name = source.col_name'
).whenMatchedUpdate(
    condition = "source.status = 'Invalid' AND target.status = 'Valid'",
    set = {
        "status": "source.status",
        "blank_count": "source.blank_count",
        "blank_percentage": "source.blank_percentage",
        "process_timestamp": "source.process_timestamp"
    }
).whenMatchedUpdate(
    condition = "source.blank_percentage != target.blank_percentage",
    set = {
        "blank_count": "source.blank_count",
        "blank_percentage": "source.blank_percentage"
    }
).whenNotMatchedInsertAll(
).execute()
```

**What it does:** Applies the FIRST matching condition. If first condition does not match, tries second condition

**Use case:**
* Different update strategies based on different conditions
* Priority-based updates
* Complex business logic

**Example:** 
* If status changes from Valid to Invalid: update ALL fields (high priority)
* Else if blank_percentage changed: update only metrics (lower priority)

---

## 5. Update with Expressions

```python
target_delta.alias('target').merge(
    df_log.alias('source'),
    'target.batch_id = source.batch_id AND target.col_name = source.col_name'
).whenMatchedUpdate(
    set = {
        "blank_count": "source.blank_count",
        "blank_percentage": "source.blank_percentage",
        "status": "CASE WHEN source.blank_percentage >= 20 THEN 'Invalid' ELSE 'Valid' END",
        "process_timestamp": "current_timestamp()",
        "update_count": "target.update_count + 1"  # Increment counter
    }
).whenNotMatchedInsertAll(
).execute()
```

**What it does:** Uses SQL expressions and functions in the SET clause

**Use case:**
* Calculated fields
* Increment counters
* Complex transformations
* Derive values from multiple columns

---

## Interview Talking Points

**Q: Why use selective update instead of updateAll?**

A: 
* **Performance** - Writing fewer columns is faster
* **Audit trail** - Preserve original timestamps or metadata
* **Immutability** - Certain columns should never change after creation
* **Data quality** - Prevent accidental overwrites of critical fields

**Q: When would you use conditional updates?**

A:
* **Change Data Capture (CDC)** - Only write when data actually changes
* **Cost optimization** - Reduce storage and compute costs on large tables
* **Compliance** - Maintain immutable audit fields
* **Performance** - Skip unnecessary writes

**Q: Can you have multiple whenMatchedUpdate clauses?**

A: Yes! They are evaluated in order, and the FIRST matching condition is applied. Think of it like a CASE WHEN statement - once a condition matches, it stops evaluating.

---

## Common Pattern: Slowly Changing Dimension (SCD) Type 1

```python
# Update current record with latest values, preserve created_at
target_delta.alias('target').merge(
    df_log.alias('source'),
    'target.batch_id = source.batch_id AND target.col_name = source.col_name'
).whenMatchedUpdate(
    set = {
        "blank_count": "source.blank_count",
        "blank_percentage": "source.blank_percentage",
        "status": "source.status",
        "updated_at": "current_timestamp()"  # Update last modified time
        # Note: created_at is NOT in the set clause, so it remains unchanged
    }
).whenNotMatchedInsert(
    values = {
        "batch_id": "source.batch_id",
        "col_name": "source.col_name",
        "blank_count": "source.blank_count",
        "blank_percentage": "source.blank_percentage",
        "status": "source.status",
        "created_at": "current_timestamp()",
        "updated_at": "current_timestamp()"
    }
).execute()
```

This pattern maintains:
* `created_at` - never changes (first insertion time)
* `updated_at` - always updates to current time
* All other fields - update with latest values

In [0]:
%sql
-- Compare the two approaches side by side
-- Run the partial update cell multiple times and observe the difference

-- Approach 1: updateAll() - ALL fields including process_timestamp get updated
SELECT 
    'Full Update (updateAll)' as approach,
    col_name,
    blank_count,
    blank_percentage,
    status,
    process_timestamp
FROM pyspark_cata_vijay.source.column_validation_log_idempotent
WHERE col_name = 'employee_id'

UNION ALL

-- Approach 2: Partial Update - ONLY blank_count and blank_percentage updated
-- process_timestamp and status remain as they were when first inserted
SELECT 
    'Partial Update (specific columns)' as approach,
    col_name,
    blank_count,
    blank_percentage,
    status,
    process_timestamp
FROM pyspark_cata_vijay.source.column_validation_log_partial_update
WHERE col_name = 'employee_id'

ORDER BY approach DESC

# Quick Reference: MERGE Update Patterns

## 1. Update ALL Columns

```python
target_delta.alias('target').merge(
    df_log.alias('source'),
    'target.id = source.id'
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
```

---

## 2. Update ONLY 2 Specific Columns

```python
target_delta.alias('target').merge(
    df_log.alias('source'),
    'target.id = source.id'
).whenMatchedUpdate(
    set = {
        "column1": "source.column1",
        "column2": "source.column2"
    }
).whenNotMatchedInsertAll().execute()
```

**Key points:**
* Only column1 and column2 are updated
* All other columns remain unchanged
* Useful for preserving audit timestamps

---

## 3. Conditional Update

```python
target_delta.alias('target').merge(
    df_log.alias('source'),
    'target.id = source.id'
).whenMatchedUpdate(
    condition = "source.value != target.value",
    set = {
        "value": "source.value",
        "updated_at": "current_timestamp()"
    }
).whenNotMatchedInsertAll().execute()
```

**Key points:**
* Only updates when condition is true
* Skips write if data has not changed
* Improves performance

---

## 4. Audit Trail Pattern (Most Common)

```python
target_delta.alias('target').merge(
    df_log.alias('source'),
    'target.id = source.id'
).whenMatchedUpdate(
    set = {
        "data_column": "source.data_column",
        "updated_at": "current_timestamp()"
        # Notice: created_at is NOT here - it stays constant
    }
).whenNotMatchedInsert(
    values = {
        "id": "source.id",
        "data_column": "source.data_column",
        "created_at": "current_timestamp()",
        "updated_at": "current_timestamp()"
    }
).execute()
```

**Key points:**
* created_at = immutable (first insertion time)
* updated_at = mutable (last modification time)
* Production-ready pattern

---

## 5. Update with SQL Expressions

```python
target_delta.alias('target').merge(
    df_log.alias('source'),
    'target.id = source.id'
).whenMatchedUpdate(
    set = {
        "count": "target.count + 1",  # Increment
        "status": "CASE WHEN source.value > 100 THEN 'HIGH' ELSE 'LOW' END",  # Conditional
        "percentage": "(source.value / target.total) * 100",  # Calculation
        "updated_at": "current_timestamp()"  # Function
    }
).whenNotMatchedInsertAll().execute()
```

**Key points:**
* Can use SQL expressions in set clause
* Can reference both source and target columns
* Can use SQL functions

---

## Interview Cheat Sheet

**Q: What is the difference between whenMatchedUpdateAll and whenMatchedUpdate?**

A:
* `whenMatchedUpdateAll()` - Updates ALL columns from source to target
* `whenMatchedUpdate(set={...})` - Updates ONLY specified columns

**Q: Can you update different columns based on different conditions?**

A: Yes! Use multiple whenMatchedUpdate clauses with different conditions. The FIRST matching condition is applied.

```python
.whenMatchedUpdate(
    condition = "source.priority = 'HIGH'",
    set = {...}  # High priority logic
).whenMatchedUpdate(
    condition = "source.priority = 'LOW'",
    set = {...}  # Low priority logic
)
```

**Q: How do you maintain an immutable created_at timestamp?**

A: Simply do NOT include created_at in the whenMatchedUpdate set clause. Only include it in whenNotMatchedInsert.

**Q: Can you use expressions in the set clause?**

A: Yes! You can use SQL expressions, functions, and reference both source and target columns.

---

## Common Mistakes to Avoid

1. **Forgetting to preserve audit columns**
   * Wrong: Include created_at in update set
   * Right: Only include created_at in insert values

2. **Using updateAll when you need selective updates**
   * Wrong: whenMatchedUpdateAll() when you want to preserve some columns
   * Right: whenMatchedUpdate(set={...}) with only the columns you want to update

3. **Not using conditions for optimization**
   * Wrong: Update every record on every run
   * Right: Add condition to only update when data actually changed

4. **Incorrect expression syntax**
   * Wrong: `"target.count + 1"` (missing source/target reference)
   * Right: `"target.count + 1"` for target column or `"source.value"` for source column

# Scen1 - Incremental Data Loading in Databricks

In [0]:
df = spark.read.format('csv').option('header', True).option('inferSchema',True).load('/Volumes/pyspark_cata_vijay/source/interview_practice_vol_1/raw_source/sales_data_first.csv')

display(df)

AutoLOader Query

In [0]:
df = spark.readStream.format('cloudFiles').option('cloudFiles.format', 'csv')\
    .option('inferSchema', True)\
        .option('cloudFiles.schemaLocation', '/Volumes/pyspark_cata_vijay/source/interview_practice_vol_1/raw_source/checkPoint')\
            .load('/Volumes/pyspark_cata_vijay/source/interview_practice_vol_1/raw_source')

In [0]:
df.writeStream.format('delta')\
    .option('cloudFiles.checkPointLocation','/Volumes/pyspark_cata_vijay/source/interview_practice_vol_1/raw_source/checkPoint')\
        .option('mergeSchema', True)\
            .trigger(once=True)\
                .start('/Volumes/pyspark_cata_vijay/source/interview_practice_vol_1/raw_source/')

# **"Scen2 - Schema Evolution in PySpark Streaming"**, the creator explains how to handle situations where new columns are added to incoming data files during an incremental load (Autoloader) process.

### Key Concepts:
* **The Problem:** When *Autoloader* encounters a file with an extra column (e.g., *return_flag*) that wasn't in the initial files, the stream will throw an **"unknown field exception"** and stop.
* **How it works:** 
    1. *Autoloader* detects the new column.
    2. It performs schema inference on the latest micro-batch.
    3. It updates the internal schema location with the new schema before throwing the error.

### How to Fix It:
1. **Restart the stream:** After the error occurs, simply restarting the query will allow it to pick up the updated schema.
2. **Destination Schema Mismatch:** You must also handle the *Delta table* (sink) side. Simply restarting isn't enough because the existing Delta table doesn't have the new column. You need to add the following option to your write stream: 
   `.option("mergeSchema", "true")`

### Additional Tips:
* **Schema Evolution Mode:** The default mode for *Autoloader* is `addNewColumns`, which explains why the stream breaks when it finds a new column.
* **Rescue Data:** The creator mentions an alternative approach called **"rescue" mode**. If you use this, *Autoloader* creates a specific column that captures all unrecognized data in a dictionary format (key-value pairs). This prevents the stream from failing even if the schema changes, as it avoids the need for manual `mergeSchema` operations.

# Scen3 - Slowly Changing Dimensions in Databricks

In [0]:
df = spark.read.format('csv').option('header', True)\
    .option('inferSchema', True)\
    .load('/Volumes/pyspark_cata_vijay/source/interview_practice_vol_1/raw_source/products_dim_table.csv')

display(df)

In [0]:
from pyspark.sql.types import *
from pyspark.sql.window import *
from pyspark.sql.functions import *
from delta.tables import *

df = df.select('p_id','p_name','p_category').filter(col('p_id').isNotNull())

display(df)

In [0]:
intial = 0

In [0]:
if (intial == 0):
    obj_delta = DeltaTable.forPath(spark,'/Volumes/pyspark_cata_vijay/source/interview_practice_vol_1/raw_source/sink')
    obj_delta.alias('trg').merge(df.alias('src'),'trg.p_id = src.p_id')\
    .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
            .execute()
else:
    df.write.format("delta").mode("append")\
        .saveAsTable("pyspark_cata_vijay.source.products_dim_new")

In [0]:
%sql
Select *  from pyspark_cata_vijay.source.products_dim_new

In [0]:
len[dbutils.fs.ls('/Volumes/pyspark_cata_vijay/source/interview_practice_vol_1/raw_source/sink')]

# How to create class having custom methods in pyspark

In [0]:
df = spark.read.format('csv').option('header', True).option('inferSchema', True).load('/Volumes/pyspark_cata_vijay/source/interview_practice_vol_1/raw_source/sales_data_first.csv')

display(df)

In [0]:
from pyspark.sql.window import *
from pyspark.sql.types import *
from pyspark.sql.functions import *

df_new = df.withColumn('dr', dense_rank().over(Window.partitionBy('Month').orderBy(col('Units_Sold').desc())))

display(df_new)

In [0]:
class WinFunc:
    df = df.withColumn('dr', dense_rank().over(Window.partitionBy('Month').orderBy(col('Units_Sold').desc())))    
    
    def dr_func(self,col_name,part_col,order_col):
        self.df = self.df.withColumn(col_name, dense_rank().over(Window.partitionBy(part_col).orderBy(col(order_col).desc())))
        return self.df

In [0]:
obj = WinFunc()

obj.df = df_new

In [0]:
df_new = obj.dr_func('new_dr','Year','Revenue')

display(df_new)

In [0]:
df = df.withColumn('case', when(col('Units_Sold')==1, lit("low")).when(col('Units_Sold')==2, lit("Med")).otherwise(lit('High'))).display()

# Find Cumulative Sum

In [0]:
df_res = df.withColumn('cum_sales', sum(col('Units_Sold')).over(Window.partitionBy('Month').orderBy('Units_Sold').rowsBetween(Window.unboundedPreceding, Window.currentRow))).display()

# Scen6 - Using Python Loops for Parametrized Notebooks

In [0]:
dbutils.widgets.text('Uni','')

In [0]:
var =dbutils.widgets.get('Uni')

print(var)

In [0]:
for i in var:
    df = spark.read.format('csv').option('header', True).option('inferSchema', True).load('/Volumes/pyspark_cata_vijay/source/interview_practice_vol_1/raw_source/sales_data_first.csv')
    df = df.filter(col('Units_Sold')==i)
    
    # df.write.format('csv').mode('append').option('path',f"/Volumes/pyspark_cata_vijay/source/interview_practice_vol_1/{i}").save()

    df.write.format("csv")\
            .mode("append")\
            .option("path",f"/Volumes/pyspark_cata_vijay/source/interview_practice_vol_1/kaka_{i}")\
            .save()



# Scen7 - Data Skewness in PySpark 

In [0]:
df = spark.read.format('csv').option('header', True).option('inferSchema', True).load('/Volumes/pyspark_cata_vijay/source/interview_practice_vol_1/raw_source/sales_data_first.csv')

df = df.withColumn('part', spark_partition_id())

df.repartition(4)

display(df)

# Scen9 - Error Handling in PySpark using Python

with the help of try & except we can handle error

In [0]:
try:
    df = spark.read.format('csv').option('header', True).option('inferSchema', True).load('/Volumes/pyspark_cata_vijay/source/interview_practice_vol_1/raw_source/sales_data_first.csv_lola')
    display(df)

except Exception as ee:
    print(f"you got error {ee}")

# Scen10 - PySpark Streaming using Delta Tables

**Scenario 10: Incremental Data Loading using Delta Tables**

The goal is to **incrementally stream data** from an existing *Delta table* into a new destination, ensuring that only new or updated records are processed. This process involves configuring **checkpointing** and managing *Delta* transaction logs to track the data versioning effectively.

# Do you know what cloud architecture is and how it functions?


## Cloud Architecture - Interview Summary

**Definition**: Cloud architecture is the design of technology components (compute, storage, networking, databases) delivered as on-demand services over the internet rather than on-premises hardware.

**Key Components**:
* **Frontend** - User interface (web/mobile apps, APIs)
* **Backend** - Cloud provider infrastructure (AWS, Azure, GCP)
* **Network** - Connection layer between frontend and backend

**How It Works**:
* **Virtualization** - Physical servers pooled into virtual resources
* **On-Demand Provisioning** - Resources allocated instantly via APIs
* **Auto-Scaling** - Dynamically adjusts capacity based on demand
* **Pay-Per-Use** - Billing based on actual consumption

**Key Benefits**:
* Eliminates upfront infrastructure investment
* Scales instantly with workload
* High availability through distributed architecture
* Global reach via multiple data centers

**Service Models**:
* **IaaS** - Virtual machines, storage (EC2, S3)
* **PaaS** - Development platforms (Databricks, Azure App Service)
* **SaaS** - Ready-to-use applications (Salesforce, Office 365)

**Real-World Example**: Databricks runs on cloud infrastructure, provisioning Spark clusters on-demand. When you run a notebook, it spins up compute, processes data, then scales down—you pay only for execution time.

This concise answer covers fundamentals, demonstrates understanding of both concepts and practical application, and shows relevance to modern data engineering.

# Suppose you received 1M records on September 5 and discovered on September 6 that one column had incorrect values. How would you backfill the data?

Would you rerun the whole pipeline?

What would be the intelligent way to backfill without rerunning the entire dataset?

How would you update only the affected column/data in your system?

## Backfilling Data with Column-Level Issues - Interview Answer

**Problem**: 1M records loaded on Sept 5, one column has bad data discovered on Sept 6.

**❌ Avoid**: Rerunning the entire pipeline (inefficient, wastes compute, impacts downstream systems)

**✅ Intelligent Approach**:

**1. Delta Lake UPDATE (Best for Databricks)**
```sql
UPDATE target_table 
SET faulty_column = corrected_value 
WHERE date = '2026-09-05' 
  AND <condition_identifying_bad_records>
```
- Only touches affected column/rows
- Leverages Delta's ACID transactions
- No full reprocessing needed

**2. MERGE Operation (For Complex Logic)**
```python
# Read corrected data for Sept 5
corrected_df = get_corrected_data('2026-09-05')

# Merge only the affected column
deltaTable.alias('target').merge(
    corrected_df.alias('source'),
    'target.id = source.id AND target.date = "2026-09-05"'
).whenMatchedUpdate(set={'faulty_column': 'source.corrected_column'}).execute()
```

**3. Time Travel + Selective Reprocessing**
- Use Delta Time Travel to access version before the bad load
- Reprocess ONLY Sept 5 data with corrected logic
- Overwrite that partition: `df.write.mode("overwrite").partitionBy("date").save()`

**Key Benefits**:
* **Partition-level**: Only reprocess Sept 5 partition, not all 1M+ records
* **Column-level**: UPDATE touches only the faulty column
* **Zero downtime**: Delta handles versioning, no table lock
* **Audit trail**: Time travel preserves history

**Interview Talking Point**: "In production, I'd choose UPDATE for simple fixes or MERGE for complex transformations, leveraging Delta's transactional capabilities to avoid full pipeline reruns and minimize compute costs."

# If the 250M records could contain duplicates, how did you deduplicate them?

In [0]:
df = df.withColumn('dedup', row_number().(Window.partitionBy('part_col').orderBy(col('order_col').desc()))).filter(col('dedup')==1)

# How did you decide whether to INSERT/APPEND or UPDATE an existing record?


Check existence: Join source and target on business key (ID)

WHEN MATCHED → Record exists → UPDATE (only if values changed to avoid unnecessary writes)

WHEN NOT MATCHED → Record doesn't exist → INSERT
Single MERGE: Both operations in one statement minimizes data scans
Change detection: Compare columns to update only when data actually changed (reduces Delta log bloat)

In [0]:
%sql
--I will use the below 1st Merge Statemnets to Update the existing expired records
Merge INTO target_table as trg
USING source_table as src
ON trg.ID = src.ID
WHEN MATCHED and
trg.col1 <> src.col1
OR
trg.col2 <> src.col2
OR
trg.col3 <> src.col3
THEN UPDATE SET
trg.col1 = src.col1,
trg.col2 = src.col2,
trg.col3 = src.col3

In [0]:
%sql
-- I will use the second merge statements to insert the new records
Merge INTO target_table as trg
USING source_table as src
ON trg.ID = src.ID
WHEN NOT MATCHED THEN INSERT *

--In this way i will insert new records and update existing records

# If 150M records belong to insurance company A and 50M each to B and C, wouldn’t the A data move to a single partition?

"Without handling skew, company A's 150M records create a single-partition bottleneck that slows the entire job"

"Salting splits the hot key across multiple partitions by adding a random suffix"

"Adaptive Query Execution in Spark 3.x+ auto-detects and mitigates skew at runtime"

"Monitor Spark UI - look for stage tasks where one task takes significantly longer (skew indicator)"

In [0]:
# Salting Technique
# Add random salt to split company A across multiple partitions
from pyspark.sql.functions import rand, concat, col

df = df.withColumn('salted_key', when(col('company_id') == 'A', concat(col('company_id'), lit('_'), (rand() * 10).cast('int')))
    .otherwise(col('company_id'))
)

# Now partition by salted_key - company A splits into 10 partitions
df_partitioned = df.repartition('salted_key')

# How would you identify whether a Spark job is skewed?

Primary Method: Spark UI (Post-Execution)

Check Stages tab → Look for tasks where one takes significantly longer (e.g., task 1 takes 600s, others take 60s)

Check Task Metrics → Uneven shuffle read/write sizes indicate skew

Monitor Spill Metrics → High spill-to-disk on specific tasks signals memory pressure from skewed partitions

Review GC Time → Excessive GC on certain executors indicates large partitions.

# Secondary Method code based profiling

In [0]:
#I will use Spark_partition_id to get the partition id over my dataset

df = df.withColumn('part', spark_partition_id())

#Then i will apply group by and count on my data set to get the count of records for each partition

df = df.groupBy('part').count()

display(df)

#in this way i will get know whether thdata is equally distributed among all the partitions or not.

# Additional Checks:
Run EXPLAIN() to see physical plan and shuffle operations
Check for skewed joins using AQE metrics (spark.sql.adaptive.skewJoin.enabled)

# What is predicate pushdown?

## Predicate Pushdown - Interview Answer

**Definition**: Predicate pushdown is a query optimization technique where filter conditions (predicates) are moved closer to the data source, reducing the amount of data read and transferred.

### How It Works:

Instead of:
1. Read entire dataset into memory
2. Apply filter conditions
3. Process remaining data

Predicate pushdown does:
1. **Push filter to the storage layer** (Parquet, Delta, JDBC source)
2. Read **only the relevant data** that matches the filter
3. Process smaller dataset

### Example:

```python
# Query with WHERE clause
df = spark.read.parquet("s3://bucket/data") \
    .filter(col("date") == "2026-09-05")
```

**Without pushdown**: Reads all Parquet files, then filters in Spark.

**With pushdown**: Parquet reader applies the filter, reads only matching row groups/pages

### Benefits:

* **Reduced I/O** - Less data read from storage
* **Lower network transfer** - Smaller data movement
* **Faster queries** - Less data to process in memory
* **Cost savings** - Fewer bytes scanned (important for cloud storage)

### When It Works:

✅ **Parquet/ORC files** - Column statistics enable skip

✅ **Delta Lake** - Partition pruning + data skipping

✅ **JDBC sources** - Filters pushed to database (PostgreSQL, MySQL)

✅ **Partitioned data** - Partition pruning is a form of pushdown

❌ **CSV/JSON** (row-based formats have limited pushdown)

❌ **Complex UDFs** (non-deterministic functions block pushdown)

### Real-World Impact:

"A query filtering `WHERE date = '2026-09-05'` on a year of daily-partitioned data only reads **1 partition** instead of **365**, reducing scan by 99.7%"

# Is predicate pushdown the work of the Catalyst Optimizer?

Yes, Predicate pushdown is one of Catalyst's logical optimizations.

Catalyst identifies filters in the query plan and pushes them to the data source layer, so Parquet/Delta readers only scan matching data. You can verify this optimization by checking df.explain() output."

The key insight: Catalyst automates these optimizations — you don't manually code pushdown, the optimizer does it for you when you write .filter() or WHERE clauses

# What is serialization?

Is serialization used when data is transferred between executors?

Is serialization used when data/task information is transferred from driver to executor?

Definition: Serialization is the process of converting an in-memory object/data structure into a byte stream format that can be transmitted over the network or stored on disk. Deserialization is the reverse process.


# 1. Is serialization used when data is transferred between executors?
Yes, absolutely. During shuffle operations (joins, groupBy, repartition), data moves between executors across the network and must be serialized.

Example scenarios:

Shuffle operations: groupBy(), reduceByKey(), join() trigger data exchange between executors
Wide transformations: Data crossing partition boundaries requires serialization
The shuffled data is serialized, sent over the network, then deserialized on the receiving executor

# 2. Is serialization used when driver sends tasks to executors?

Yes. The driver serializes task information (closures, functions, variables) and sends it to executors.

What gets serialized:

Task code/closures: The function you want to execute (e.g., map(), filter() logic)
Broadcast variables: Efficiently distributed to all executors
Task metadata: Partition info, dependencies

# For a 10GB CSV, when using spark.read.csv() and df.show(), what exactly happens?
---
## ✅ Complete Interview Answer:

**Question**: For a 10GB CSV, when using `spark.read.csv()` and `df.show()`, what exactly happens?

### Step-by-Step Execution:

**1. spark.read.csv() - Lazy Transformation**
```python
df = spark.read.csv("s3://bucket/10GB_file.csv")
```
* **Creates a logical plan** (blueprint) but doesn't read any data yet
* **No data movement** - just metadata about the file location and schema
* Returns a DataFrame reference immediately (even for 10GB file)

**2. df.show() - Action Triggered**
```python
df.show()  # Default: 20 rows
```

**What Actually Happens:**

**a) Catalyst Optimizer Creates Physical Plan:**
* Analyzes logical plan
* Applies optimizations (predicate pushdown, column pruning)
* Generates physical execution plan

**b) Driver Distributes Tasks:**
* Divides 10GB CSV into **partitions** (e.g., 128MB chunks = ~80 partitions)
* Sends tasks to executors: "Read your partition and return first N rows"

**c) Executors Read Partial Data:**
* **Critical Point**: Executors don't read entire 10GB
* They read **only until 20 rows are collected**
* Might read only the **first few partitions** (e.g., first 256MB instead of 10GB)
* Uses **limit pushdown** optimization

**d) Results Aggregated:**
* Executors send 20 rows back to driver
* Driver displays results
* **Rest of the 10GB is never touched**

---

### Performance Implications:

| Operation | Data Read | Time |
| --- | --- | --- |
| `spark.read.csv()` | **0 bytes** (lazy) | Instant |
| `df.show(20)` | **~256MB** (first few partitions) | Seconds |
| `df.count()` | **Full 10GB** (scans entire file) | Minutes |

---

## Interview Talking Points:

**"When you call `spark.read.csv()` on a 10GB file, Spark creates a logical plan without reading any data - it's lazy evaluation. When `df.show()` triggers the action, Spark doesn't process the entire 10GB. It uses limit pushdown to read only the first few partitions until it collects 20 rows - maybe 256MB instead of 10GB. This is why `show()` is fast even on huge files, while operations like `count()` that require scanning the entire dataset take much longer."**

---

### Fixed Version of Your Answer:

"When we use `spark.read.csv()`, Spark creates a **logical execution plan** without reading any data (lazy evaluation). When we call `df.show()`, it triggers an **action** that causes Spark to execute the plan. However, `df.show()` only reads **enough partitions to collect 20 rows** (not the entire 10GB), using limit pushdown optimization. The driver distributes tasks to executors, who read their assigned partitions and return rows until the limit is reached."

# Is the data read by executors only?
# 
# If filtering happens on the executors, why would the driver transfer data to the executors?
# 
# Is it correct to say that the driver serializes the data and sends it to the executor?

## Follow-Up Questions - Clarifying Driver vs Executor Roles

Let me address these critical misconceptions about Spark's execution model:

---

### 1. **Is the data read by executors only?**

**✅ YES - Executors read the data directly from storage.**

* **Executors** connect to the data source (S3, HDFS, Azure Blob) and read the CSV data
* **Driver** never touches the raw 10GB CSV file
* Driver only receives **aggregated results** (e.g., 20 rows for `show()`)

---

### 2. **If filtering happens on the executors, why would the driver transfer data to the executors?**

**This is the key misconception! The driver does NOT transfer data to executors.**

**What Actually Happens:**

**Driver sends:**
* **Task instructions** (serialized code: "read partition 1, filter where X, return first N rows")
* **Metadata** (file path, partition boundaries, schema)
* **Execution plan** (how to process the data)

**Executors:**
* **Read data directly from storage** (S3/HDFS) using the file path
* Apply filters locally
* Send only **results** back to driver

**Data Flow:**
```
Storage (S3) → Executors (read & process) → Driver (receives results only)
```

**NOT:**
```
Storage → Driver → Executors ❌ WRONG!
```

---

### 3. **Is it correct to say that the driver serializes the data and sends it to the executor?**

**❌ NO - This is INCORRECT.**

**What the driver actually serializes and sends:**
* **Task code** (the lambda/function to execute)
* **Closures** (variables captured in functions)
* **Broadcast variables** (small lookup tables shared across executors)
* **Partition metadata** (which partition each executor should read)

**The driver does NOT serialize or send the 10GB CSV data itself.**

---

## Correct Data Flow for `spark.read.csv()` + `df.show()`

### Step-by-Step:

**1. Driver's Role:**
```
Driver:
  ├─ Creates logical plan
  ├─ Optimizes with Catalyst
  ├─ Generates physical plan
  ├─ Serializes TASK CODE: "read partition X, take 20 rows"
  └─ Sends task to executors
```

**2. Executors' Role:**
```
Executors:
  ├─ Receive task code from driver
  ├─ Deserialize task instructions
  ├─ Read data DIRECTLY from S3 (file path in task)
  ├─ Process data locally (filter, transform)
  ├─ Collect requested rows (20 rows)
  └─ Send results back to driver
```

**3. Driver Receives:**
```
Driver:
  └─ Collects 20 rows from executors
  └─ Displays to user
```

---

## Interview Talking Point - The Critical Distinction:

**"The driver NEVER reads or transfers the raw data. It serializes the TASK CODE (instructions) and sends it to executors. Executors read data directly from storage (S3, HDFS) using the file path provided in the task. This distributed read architecture is what makes Spark scalable - if the driver had to read and distribute 10GB, it would become a bottleneck. Only task instructions flow from driver to executors, and only results (20 rows) flow back."**

## Spark Join Strategies - Interview Answers

### 1. **Do you know about join strategies in Spark?**

Yes, Spark uses different join strategies to optimize how two datasets are joined based on their size, distribution, and join keys. The strategy chosen impacts performance significantly.

---

### 2. **What are the three join strategies?**

**1. Broadcast Hash Join (BHJ)**
* Small dataset (< 10MB default) is broadcast to all executors
* No shuffle required
* Fastest join strategy

**2. Sort-Merge Join (SMJ)**
* Both datasets are sorted by join key, then merged
* Requires shuffle of both datasets
* Default for large-large joins

**3. Shuffle Hash Join (SHJ)**
* Shuffles both datasets, builds hash table from one side
* Used when sort-merge is not applicable (e.g., non-equi joins)
* Less common, requires specific conditions

---

### 3. **If both datasets are large, which join strategy would you use?**

**Sort-Merge Join (SMJ)** - This is the default and most efficient strategy for large-large joins.

**Why:**
* Handles large datasets that can't fit in memory
* Leverages sorting to avoid building large hash tables
* Works well with partitioned data

---

### 4. **If both datasets are large, won't the shuffle slow down the operation?**

**Yes, shuffle does add overhead, but it's unavoidable for large-large joins.**

**Mitigations:**
* **Pre-partition data** by join key to avoid shuffle
* **Use bucketing** - if tables are bucketed on join key, shuffle is eliminated
* **Enable AQE** (Adaptive Query Execution) to optimize shuffle dynamically
* **Tune shuffle partitions**: `spark.sql.shuffle.partitions` (default 200)

**Reality:** Shuffle cost is acceptable because the alternative (broadcasting large datasets) would cause executor OOM errors.

---

### 6. **Does high cardinality of the join key affect the sorting/join?**

**Yes, high cardinality is GOOD for Sort-Merge Join:**

**High Cardinality (millions of unique keys):**
* ✅ Data distributes evenly across partitions
* ✅ No data skew
* ✅ Efficient parallel processing
* ✅ Sorting is efficient within partitions

**Low Cardinality (few unique keys):**
* ❌ Data skew - most records go to few partitions
* ❌ Some executors overloaded, others idle
* ❌ Sorting large skewed partitions is slow
* **Solution**: Use salting to split hot keys

**Interview Point:** "High cardinality reduces skew and improves Sort-Merge Join performance by distributing data evenly across partitions."

---

### 7. **Should you use Sort-Merge Join or Hash Join?**

**Decision Matrix:**

| Scenario | Use |
| --- | --- |
| One dataset **< 10MB** | **Broadcast Hash Join** (no shuffle) |
| Both datasets **large** | **Sort-Merge Join** (default) |
| Data **already sorted/bucketed** on join key | **Sort-Merge Join** (skip shuffle) |
| High **data skew** | **Broadcast Hash Join** (if small side) or **Salting + Sort-Merge** |
| **Non-equi joins** (>, <, !=) | **Shuffle Hash Join** or nested loop |

**General Rule:**
* **Broadcast Hash Join** = Small-Large joins (fastest, no shuffle)
* **Sort-Merge Join** = Large-Large joins (scalable, handles skew with tuning)

---

### 8. **What join strategy does Spark use by default?**

**Spark decides automatically using this hierarchy:**

**1. Broadcast Hash Join** (if one side < `spark.sql.autoBroadcastJoinThreshold`)
* Default threshold: **10MB**
* Checked first for every join

**2. Sort-Merge Join** (if broadcast not applicable)
* Default for large-large joins
* Most common in production

**3. Shuffle Hash Join / Cartesian Join** (rare, last resort)

**You can verify with:**
```python
df1.join(df2, "key").explain()
# Look for "BroadcastHashJoin" or "SortMergeJoin"
```

---

### 9. **What configuration controls which join strategy Spark uses?**

**Key Configurations:**

**1. Broadcast Join Threshold:**
```python
# Default: 10MB (10485760 bytes)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "20971520")  # 20MB
# Set to -1 to disable broadcast join
```

**2. Shuffle Partitions:**
```python
# Default: 200
spark.conf.set("spark.sql.shuffle.partitions", "400")
```

**3. Adaptive Query Execution (AQE) - Auto-optimizes joins:**
```python
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
```

**4. Force Specific Join:**
```python
# Force broadcast
df1.join(broadcast(df2), "key")

# Disable broadcast for testing
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
```

---

## Interview Talking Point Summary:

**"Spark has three main join strategies: Broadcast Hash Join (for small-large joins, no shuffle), Sort-Merge Join (default for large-large joins, requires shuffle and sort), and Shuffle Hash Join (rare, for specific cases). By default, Spark uses Broadcast Hash Join if one dataset is under 10MB (controlled by `spark.sql.autoBroadcastJoinThreshold`), otherwise it falls back to Sort-Merge Join. In Sort-Merge Join, data is hash-partitioned by join key, sorted within each partition, then merged. High cardinality is beneficial as it distributes data evenly and reduces skew. For production large-large joins, I rely on Sort-Merge Join with AQE enabled to handle skew automatically."**

---

%md
## Sort-Merge Join: How Spark Sorts the Joining Data

### Visual Explanation

---

### **Phase 1: SHUFFLE (Partitioning by Join Key)**

**Original Datasets (Unsorted, Distributed Randomly):**

```
Dataset A (Orders):                Dataset B (Customers):
customer_id | order_id            customer_id | name
100         | A1                  300         | John
200         | A2                  100         | Alice
100         | A3                  200         | Bob
300         | A4
```

**↓ Hash Partitioning on customer_id ↓**

**After Shuffle (Same Keys Go to Same Partition):**

```
┌─────────────────────────────────────────────────────────────────┐
│ Partition 1 (Executor 1) - customer_id = 100                    │
├─────────────────────────────────────────────────────────────────┤
│ Dataset A: [100, A1], [100, A3]                                 │
│ Dataset B: [100, Alice]                                         │
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────┐
│ Partition 2 (Executor 2) - customer_id = 200                    │
├─────────────────────────────────────────────────────────────────┤
│ Dataset A: [200, A2]                                            │
│ Dataset B: [200, Bob]                                           │
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────┐
│ Partition 3 (Executor 3) - customer_id = 300                    │
├─────────────────────────────────────────────────────────────────┤
│ Dataset A: [300, A4]                                            │
│ Dataset B: [300, John]                                          │
└─────────────────────────────────────────────────────────────────┘
```

**Key Point:** All records with the SAME customer_id are now co-located on the SAME executor.

---

### **Phase 2: SORT Within Each Partition**

**Each executor sorts its partition independently by customer_id:**

```
┌─────────────────────────────────────────────────────────────────┐
│ Partition 1 (Executor 1) - SORTED                               │
├─────────────────────────────────────────────────────────────────┤
│ Dataset A: [100, A1], [100, A3]  ← already sorted               │
│ Dataset B: [100, Alice]          ← already sorted               │
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────┐
│ Partition 2 (Executor 2) - SORTED                               │
├─────────────────────────────────────────────────────────────────┤
│ Dataset A: [200, A2]             ← sorted                       │
│ Dataset B: [200, Bob]            ← sorted                       │
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────┐
│ Partition 3 (Executor 3) - SORTED                               │
├─────────────────────────────────────────────────────────────────┤
│ Dataset A: [300, A4]             ← sorted                       │
│ Dataset B: [300, John]           ← sorted                       │
└─────────────────────────────────────────────────────────────────┘
```

**Key Point:** Both datasets are now SORTED by join key within each partition.

---

### **Phase 3: MERGE (Join the Sorted Data)**

**Each executor performs a merge-sort-like join on its sorted partitions:**

```
Partition 1:
  Read from A: customer_id=100 → Match with B: customer_id=100
  ✓ Join: [100, A1, Alice]
  Read next from A: customer_id=100 → Match with B: customer_id=100
  ✓ Join: [100, A3, Alice]

Partition 2:
  Read from A: customer_id=200 → Match with B: customer_id=200
  ✓ Join: [200, A2, Bob]

Partition 3:
  Read from A: customer_id=300 → Match with B: customer_id=300
  ✓ Join: [300, A4, John]
```

**Final Output (All Partitions Combined):**

```
customer_id | order_id | name
100         | A1       | Alice
100         | A3       | Alice
200         | A2       | Bob
300         | A4       | John
```

---

### **Summary: Three Critical Steps**

1. **SHUFFLE** → Move data so matching join keys are co-located (across network)
2. **SORT** → Arrange data within each partition by join key (on each executor)
3. **MERGE** → Walk through sorted lists and join matching records (efficient linear scan)

---

### **Interview Talking Point:**

*"In Sort-Merge Join, sorting happens AFTER shuffling, WITHIN each partition. The shuffle ensures all records with the same join key land on the same executor. Then each executor sorts its partition independently. Finally, the sorted partitions are merged efficiently, like merging two sorted arrays. This approach scales to large datasets because sorting happens in parallel across all executors."*

TODO 1: Remove duplicate transactions based on
(customer_id, transaction_date, product_id, amount)

deduped_df = ...

TODO 2: Derive transaction_year from transaction_date (yyyy-MM-dd)

with_year_df = ...

TODO 3: Compute total revenue per customer
(and year if needed for partitioning)

revenue_df = ...

TODO 4: Identify top 10 customers by total revenue
(make tie-break deterministic)

top10_df = ...

TODO 5: Write the result as a Delta table partitioned by transaction_year

Example target: "finance_analytics.top10_customers_revenue"
(top10_df.write.format("delta").....)

In [0]:
# Sample input data (includes a deliberate duplicate)
data = [
   ("C001", "2024-01-15", "P01", 100.0),
   ("C001", "2024-01-15", "P01", 100.0),  # duplicate
   ("C002", "2024-02-10", "P03", 250.0),
   ("C003", "2024-02-10", "P04", 250.0),
   ("C001", "2023-12-20", "P02", 75.5),
   ("C002", "2023-11-05", "P01", 10.0),
]
sales_df = spark.createDataFrame(
   data,
   ["customer_id", "transaction_date", "product_id", "amount"]
)

In [0]:
# TODO 1: Remove duplicate transactions based on
# (customer_id, transaction_date, product_id, amount)
#display(sales_df)

from pyspark.sql.window import *
from pyspark.sql.types import *
from pyspark.sql.functions import *

deduped_df = sales_df.dropDuplicates(['customer_id','transaction_date','product_id','amount'])

display(deduped_df)

In [0]:
#TODO 2: Derive transaction_year from transaction_date (yyyy-MM-dd) with_year_df = ...

with_year_df = deduped_df.withColumn('transaction_year', year(col('transaction_date')))

display(with_year_df)

In [0]:
#TODO 3: Compute total revenue per customer (and year if needed for partitioning)

revenue_df = with_year_df.groupBy('customer_id','transaction_year').agg(sum('amount').alias('total_revenue'))\
    .orderBy(col('total_revenue').desc(),col('customer_id').asc())

display(revenue_df)

In [0]:
#TODO 4: Identify top 10 customers by total revenue (make tie-break deterministic)

top10_df = revenue_df.orderBy(col('total_revenue').desc()).limit(3)

display(top10_df)

In [0]:
#TODO 5: Write the result as a Delta table partitioned by transaction_year

#Example target: "finance_analytics.top10_customers_revenue" (top10_df.write.format("delta").....)

top10_df.write.format('delta').mode('overwrite').partitionBy('transaction_year').option('path', 'abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/{destination_folder_name}').save()

In [0]:
# =====================================================================
# COMPLETE SOLUTION FOR ONLINE PYTHON IDE (local Spark mode)
# =====================================================================
# In a Databricks notebook, `spark` is pre-created. In an online IDE,
# you must create the SparkSession yourself with master('local[*]').
# =====================================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, year

# --- 1. Create SparkSession (NOT needed in Databricks) ---
# In Databricks, `spark` is already created by the environment.
# In an online IDE, you create it yourself with master('local[*]').
try:
    spark  # already exists in Databricks?
except NameError:
    spark = SparkSession.builder \
        .appName("Interview Practice") \
        .master("local[*]") \
        .getOrCreate()

# --- Sample input data ---
data = [
    ("C001", "2024-01-15", "P01", 100.0),
    ("C001", "2024-01-15", "P01", 100.0),  # duplicate
    ("C002", "2024-02-10", "P03", 250.0),
    ("C003", "2024-02-10", "P04", 250.0),
    ("C001", "2023-12-20", "P02", 75.5),
    ("C002", "2023-11-05", "P01", 10.0),
]
sales_df = spark.createDataFrame(
    data, ["customer_id", "transaction_date", "product_id", "amount"]
)

# --- TODO 1: Remove duplicates ---
deduped_df = sales_df.dropDuplicates(
    ["customer_id", "transaction_date", "product_id", "amount"]
)
deduped_df.show()

# --- TODO 2: Derive transaction_year ---
with_year_df = deduped_df.withColumn(
    "transaction_year", year(col("transaction_date"))
)
with_year_df.show()

# --- TODO 3: Total revenue per customer + year ---
revenue_df = with_year_df.groupBy("customer_id", "transaction_year").agg(
    sum("amount").alias("total_revenue")
).orderBy(col("total_revenue").desc(), col("customer_id").asc())
revenue_df.show()

# --- TODO 4: Top 10 customers by revenue (deterministic tie-break) ---
# The orderBy in TODO 3 already ensures a deterministic order,
# so limit(10) here is safe.
top10_df = revenue_df.limit(10)
top10_df.show()

# --- TODO 5: Write as Delta table partitioned by transaction_year ---
# In an online IDE there is no Unity Catalog. Two options:
#   (a) If PySpark + delta-spark are installed, write to local disk:
#       top10_df.write.format("delta").mode("overwrite") \
#           .partitionBy("transaction_year") \
#           .save("/tmp/top10_customers_revenue")
#   (b) If Delta is not available, fall back to Parquet (same API):
#       top10_df.write.format("parquet").mode("overwrite") \
#           .partitionBy("transaction_year") \
#           .save("/tmp/top10_customers_revenue")
#
# In Databricks (where you have a catalog):
#   top10_df.write.format("delta").mode("overwrite") \
#       .partitionBy("transaction_year") \
#       .saveAsTable("finance_analytics.top10_customers_revenue")
#
# --- The write is commented out so this cell runs in Databricks too ---
# --- Uncomment the relevant block when running in an online IDE ---

# top10_df.write.format("parquet").mode("overwrite") \
#     .partitionBy("transaction_year") \
#     .save("/tmp/top10_customers_revenue")

# Verify the written data
# spark.read.parquet("/tmp/top10_customers_revenue").show()

You receive a daily file:

customer_id,
name,
email,
phone,
address

Requirements:

- Remove duplicate customers.
- Keep the latest record.
- Reject records where customer_id is null.
- Reject records where email is invalid.
- Separate valid and invalid records.
- Store valid records as Delta.
- Store rejected records separately.
- Make the pipeline idempotent.

In [0]:
from pyspark.sql.functions import *
from datetime import datetime

# Sample data with various scenarios:
# - Duplicates (same customer_id with different timestamps)
# - Null customer_id (should be rejected)
# - Invalid emails (should be rejected)
# - Valid records
data = [
    # Valid records
    ("C001", "Alice Johnson", "alice.johnson@email.com", "555-0101", "123 Main St, NY", "2024-01-15 10:00:00"),
    ("C002", "Bob Smith", "bob.smith@email.com", "555-0102", "456 Oak Ave, CA", "2024-01-15 11:00:00"),
    ("C003", "Carol White", "carol.white@email.com", "555-0103", "789 Pine Rd, TX", "2024-01-15 12:00:00"),
    
    # Duplicate customer_id (older record - should be dropped, keep latest)
    ("C001", "Alice J.", "alice.old@email.com", "555-9999", "Old Address", "2024-01-10 09:00:00"),
    
    # Duplicate customer_id (latest record - should be kept)
    ("C002", "Bob Smith Updated", "bob.smith.new@email.com", "555-0102", "789 New Street, CA", "2024-01-16 14:00:00"),
    
    # Null customer_id (should be rejected)
    (None, "David Brown", "david.brown@email.com", "555-0104", "321 Elm St, FL", "2024-01-15 13:00:00"),
    
    # Invalid emails (should be rejected)
    ("C004", "Emma Davis", "invalid-email", "555-0105", "654 Maple Dr, WA", "2024-01-15 14:00:00"),
    ("C005", "Frank Wilson", "no-at-sign.com", "555-0106", "987 Cedar Ln, OR", "2024-01-15 15:00:00"),
    ("C006", "Grace Lee", "missing@domain", "555-0107", "147 Birch Ct, NV", "2024-01-15 16:00:00"),
    
    # More valid records
    ("C007", "Henry Taylor", "henry.taylor@email.com", "555-0108", "258 Willow Way, AZ", "2024-01-15 17:00:00"),
    ("C008", "Ivy Martinez", "ivy.martinez@email.com", "555-0109", "369 Spruce Pl, CO", "2024-01-15 18:00:00"),
    
    # Another duplicate (same customer_id, middle timestamp)
    ("C007", "Henry T.", "henry.old@email.com", "555-9998", "Old Henry Address", "2024-01-12 10:00:00"),
]

customer_df = spark.createDataFrame(
    data,
    ["customer_id", "name", "email", "phone", "address", "timestamp"]
)

print(f"Total records in raw file: {customer_df.count()}")
display(customer_df.orderBy("customer_id", "timestamp"))

In [0]:
# MY SOLUTION - VIJAY

# Remove duplicate customers.
# Keep the latest record.
# Reject records where customer_id is null.
# Reject records where email is invalid.
# Separate valid and invalid records.
# Store valid records as Delta.
# Store rejected records separately.
# Make the pipeline idempotent.

from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

# Remove duplicate customers.
# Keep the latest record.
df_cleaned = customer_df.withColumn('dedup',row_number().over(Window.partitionBy('customer_id').orderBy(col('timestamp').desc()))).filter(col('dedup')==1).drop('dedup')

# Reject records where customer_id is null.
df_cleaned = df_cleaned.filter(col('customer_id').isNotNull())

#keep records where customer_id is null
df_uncleaned = customer_df.filter(col('customer_id').isNull())

# Validate email format: must contain @ and domain with .
# Pattern: at least one char before @, then @, then domain with at least one .
email_pattern = r'^[^@]+@[^@]+\.[^@]+$'

df_cleaned = df_cleaned.withColumn('valid_email',col('email').rlike(email_pattern)).filter(col('valid_email')==True).drop('valid_email')

#keep records where email is invalid
df_invalid_email = customer_df.withColumn('valid_email',col('email').rlike(email_pattern)).filter(col('valid_email')==False).drop('valid_email')

#Merging invalid records
invalid_rec = df_uncleaned.union(df_invalid_email)



# #writing valid records
# df_cleaned.write.format('delta').mode('append').option('path','abfss://container_name@storage_account_name.dfs.core.windows.net/valid').save()

#writing invalid records
invalid_rec.write.format('delta').mode('append').saveAsTable('pyspark_cata_vijay.source.INVALID_customers_prac')

from delta.tables import *
if spark.catalog.tableExists('pyspark_cata_vijay.source.customers_prac'):
    obj = DeltaTable.forName(spark,'pyspark_cata_vijay.source.customers_prac')
    obj.alias('trg').merge(df_cleaned.alias('src'), 'trg.customer_id = src.customer_id').whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

    df_cleaned.write.format('delta').mode('overwrite').saveAsTable('pyspark_cata_vijay.source.customers_prac')
else:
    df_cleaned.write.format('delta').mode('overwrite').saveAsTable('pyspark_cata_vijay.source.customers_prac')
    
in_valid_records = invalid_rec.count()
Valid_records = df_cleaned.count()

print(f"Valid Records = {Valid_records}")
display(df_cleaned)
print(f"Invalid Records = {in_valid_records}")
display(invalid_rec)



Q7. Find records present in one DataFrame but not another

In [0]:
# 1st method
result = df1.subtract(df2)

# 2nd Method
result = df1.join(df2,df1['customer_id']==df2['customer_id'],how="left_anti")

## Scenario 8: Late-Arriving Data Handling

### Problem Statement:
You have a daily sales transaction system that receives data throughout the day. However, some transactions arrive late (after the daily batch has run). You need to:

1. Process daily transactions and create a summary by store and product
2. Handle late-arriving transactions from previous dates
3. Update the summary table when late data arrives
4. Maintain an audit trail of when records were updated
5. Ensure idempotency (running the same data multiple times shouldn't create duplicates)

### Business Rules:
* Late transactions are identified by their `transaction_date` being before today's processing date
* When updating existing aggregates, add the new amounts to existing totals
* Track both `created_at` (first insert) and `updated_at` (last update) timestamps
* The summary should show: store_id, product_id, transaction_date, total_quantity, total_amount

### Sample Data:
* Two batches of data will be provided
* Batch 1: Regular daily transactions
* Batch 2: Contains some late-arriving transactions from previous dates

In [0]:
from pyspark.sql.functions import *
from datetime import datetime

# Batch 1: Current day transactions (2024-03-15)
batch1_data = [
    ("TXN001", "STORE01", "PROD001", "2024-03-15", 10, 500.0, "2024-03-15 10:30:00"),
    ("TXN002", "STORE01", "PROD002", "2024-03-15", 5, 250.0, "2024-03-15 11:00:00"),
    ("TXN003", "STORE02", "PROD001", "2024-03-15", 8, 400.0, "2024-03-15 12:00:00"),
    ("TXN004", "STORE02", "PROD003", "2024-03-15", 15, 750.0, "2024-03-15 13:00:00"),
    ("TXN005", "STORE03", "PROD002", "2024-03-15", 12, 600.0, "2024-03-15 14:00:00"),
]

batch1_df = spark.createDataFrame(
    batch1_data,
    ["transaction_id", "store_id", "product_id", "transaction_date", "quantity", "amount", "received_at"]
)

print("Batch 1 - Current Day Transactions:")
display(batch1_df)

In [0]:
# Batch 2: Contains new transactions + late-arriving transactions from previous dates
batch2_data = [
    # New transaction for today (2024-03-16)
    ("TXN006", "STORE01", "PROD001", "2024-03-16", 7, 350.0, "2024-03-16 09:00:00"),
    ("TXN007", "STORE02", "PROD002", "2024-03-16", 20, 1000.0, "2024-03-16 10:00:00"),
    
    # Late-arriving transactions (should update previous date aggregates)
    ("TXN008", "STORE01", "PROD001", "2024-03-15", 3, 150.0, "2024-03-16 11:00:00"),  # Late for 03-15
    ("TXN009", "STORE02", "PROD003", "2024-03-15", 5, 250.0, "2024-03-16 11:30:00"),  # Late for 03-15
    ("TXN010", "STORE03", "PROD001", "2024-03-14", 10, 500.0, "2024-03-16 12:00:00"), # Late for 03-14
]

batch2_df = spark.createDataFrame(
    batch2_data,
    ["transaction_id", "store_id", "product_id", "transaction_date", "quantity", "amount", "received_at"]
)

print("Batch 2 - Transactions with Late Arrivals:")
display(batch2_df)

In [0]:
# Process daily transactions and create a summary by store and product
# Handle late-arriving transactions from previous dates
# Update the summary table when late data arrives
# Maintain an audit trail of when records were updated
# Ensure idempotency (running the same data multiple times shouldn't create duplicates)

# Writing the daily batch data to a Delta Table
batch1_df = batch1_df.withColumn('updated_at',lit(None).cast('timestamp'))
batch1_df = batch1_df.withColumn('received_at',to_timestamp('received_at'))
batch1_df = batch1_df.withColumn('created_at',current_timestamp())
batch1_df.write.format('delta').mode('overwrite').saveAsTable('real_time_catalog.source.daily_batch_df')

In [0]:
# Adding a column created_at with value as current_timestamp in late arrival data
incremental_df =batch2_df.withColumn('created_at',current_timestamp())

In [0]:
incremental_df.createOrReplaceTempView("src")

In [0]:
# VIA PYSPARK

from delta.tables import *

dlt_obj = DeltaTable.forName(spark,'real_time_catalog.source.daily_batch_df')

dlt_obj.alias('trg').merge(incremental_df.alias('src'),'''trg.store_id = src.store_id AND 
                           trg.product_id = src.product_id AND 
                           trg.transaction_id = src.transaction_id''').whenMatchedUpdate(set={"trg.transaction_date" : "src.transaction_date","trg.quantity" : "src.quantity","trg.amount" : "src.amount","trg.updated_at" : current_timestamp(),"trg.received_at" : "src.received_at"})\
                               .whenNotMatchedInsert(values={"trg.transaction_id":"src.transaction_id","trg.store_id" : "src.store_id","trg.product_id":"src.product_id","trg.transaction_date" : "src.transaction_date","trg.quantity" : "src.quantity" ,"trg.amount":"src.amount","trg.received_at":"src.received_at","trg.created_at":"src.created_at"}).execute()

In [0]:
%sql
--VIA SQL

MERGE INTO real_time_catalog.source.daily_batch_df as trg
USING src
ON 
trg.store_id = src.store_id AND
trg.product_id = src.product_id AND
trg.transaction_id = src.transaction_id
WHEN MATCHED THEN UPDATE SET
trg.transaction_date = src.transaction_date,
trg.quantity = src.quantity,
trg.amount = src.amount,
trg.updated_at=current_timestamp(),
trg.received_at = src.received_at
WHEN NOT MATCHED THEN INSERT
(trg.transaction_id, trg.store_id,trg.product_id,trg.transaction_date,trg.quantity,trg.amount,trg.received_at,trg.created_at)
VALUES
(src.transaction_id, src.store_id,src.product_id,src.transaction_date,src.quantity,src.amount,src.received_at,src.created_at)

In [0]:
%sql
--The summary should show: store_id, product_id, transaction_date, total_quantity, total_amount
Select store_id,product_id,transaction_date, sum(quantity) as total_qty,sum(amount) as total_amt 
from real_time_catalog.source.daily_batch_df
group by all;

## Scenario 9: Slowly Changing Dimension Type 2 (SCD2)

### Problem Statement:
You're managing a customer dimension table that tracks historical changes. Implement SCD Type 2 logic to:

1. Insert new customers
2. Close old records (set `end_date` and `is_current = False`) when attributes change
3. Insert new versions with updated attributes
4. Keep records unchanged if nothing changed
5. Track effective dates (`start_date`, `end_date`)
6. Use `is_current` flag for the active record

### Attributes to Track:
* customer_id (business key - doesn't change)
* name, email, city, membership_tier (tracking these for changes)

### Business Rules:
* If ANY tracked attribute changes, create a new version
* end_date for current records is '9999-12-31'
* New records get today's date as start_date
* Closed records get today's date as end_date

### Sample Data:
* Initial Load: Current customer records
* Delta Load: Contains updates, new customers, and unchanged records

In [0]:
# Initial customer dimension (existing records)
initial_customers = [
    ("CUST001", "Alice Johnson", "alice@email.com", "New York", "Gold", "2024-01-01", "9999-12-31", True, 1),
    ("CUST002", "Bob Smith", "bob@email.com", "Los Angeles", "Silver", "2024-01-01", "9999-12-31", True, 1),
    ("CUST003", "Carol White", "carol@email.com", "Chicago", "Bronze", "2024-01-01", "9999-12-31", True, 1),
    ("CUST004", "David Brown", "david@email.com", "Houston", "Gold", "2024-02-01", "9999-12-31", True, 1),
]

current_dim_df = spark.createDataFrame(
    initial_customers,
    ["customer_id", "name", "email", "city", "membership_tier", "start_date", "end_date", "is_current", "version"]
)

print("Current Customer Dimension:")
display(current_dim_df)

In [0]:
# Incoming delta (new feed with updates)
delta_customers = [
    # Unchanged record
    ("CUST001", "Alice Johnson", "alice@email.com", "New York", "Gold"),
    
    # Changed: city updated (Los Angeles -> San Francisco)
    ("CUST002", "Bob Smith", "bob@email.com", "San Francisco", "Silver"),
    
    # Changed: membership_tier upgraded (Bronze -> Silver)
    ("CUST003", "Carol White", "carol@email.com", "Chicago", "Silver"),
    
    # Changed: email and membership tier (Gold -> Platinum)
    ("CUST004", "David Brown", "david.brown.new@email.com", "Houston", "Platinum"),
    
    # New customer (doesn't exist in dimension)
    ("CUST005", "Emma Davis", "emma@email.com", "Phoenix", "Bronze"),
]

delta_df = spark.createDataFrame(
    delta_customers,
    ["customer_id", "name", "email", "city", "membership_tier"]
)

print("Incoming Delta (New Customer Feed):")
display(delta_df)

print("\n📋 Expected Outcome:")
print("  • CUST001: No change (keep existing record)")
print("  • CUST002: Close old record, insert new version with San Francisco")
print("  • CUST003: Close old record, insert new version with Silver tier")
print("  • CUST004: Close old record, insert new version with new email + Platinum")
print("  • CUST005: Insert new record (version 1)")

In [0]:
%sql
DROP TABLE IF EXISTS pyspark_cata_vijay.source.current_dim_df

In [0]:
delta_df.createOrReplaceTempView('src') #source

current_dim_df.write.format('delta').mode('overwrite').saveAsTable('pyspark_cata_vijay.source.current_dim_df') #Target

In [0]:
%sql
-- STRING to DATE type change is not supported via ALTER TABLE ALTER COLUMN
-- Use CREATE OR REPLACE TABLE AS SELECT to recreate with the desired types

CREATE OR REPLACE TABLE pyspark_cata_vijay.source.current_dim_df AS
SELECT
  customer_id,
  name,
  email,
  city,
  membership_tier,
  to_date(start_date) AS start_date,
  to_date(end_date) AS end_date,
  is_current,
  version
FROM pyspark_cata_vijay.source.current_dim_df;

Select * from pyspark_cata_vijay.source.current_dim_df;


In [0]:
%sql
-- 1st Marking the older records as expired
Merge into pyspark_cata_vijay.source.current_dim_df as trg
USING src
ON trg.customer_id = src.customer_id
AND trg.is_current = 'true'
WHEN MATCHED AND
src.name <> trg.name OR
src.email <> trg.email OR
src.city <> trg.city OR
src.membership_tier <> trg.membership_tier
THEN UPDATE SET
trg.is_current = 'false',
trg.end_date = current_date(),
trg.version = 1-trg.version

In [0]:
%sql
--2nd Merge statement to insert new and updated records
Merge into pyspark_cata_vijay.source.current_dim_df as trg
USING src
ON trg.customer_id = src.customer_id
AND trg.is_current = 'true'
WHEN NOT MATCHED THEN INSERT
(trg.customer_id,trg.name,trg.email,trg.city,trg.membership_tier,trg.start_date,trg.end_date,trg.is_current,trg.version)
VALUES
(src.customer_id,src.name,src.email,src.city,src.membership_tier,current_date(),'9999-12-31','true',1)

In [0]:
%sql
Select * from pyspark_cata_vijay.source.current_dim_df

In [0]:
# Attributes to Track:
# customer_id (business key - doesn't change)
# name, email, city, membership_tier (tracking these for changes)


from pyspark.sql.types import *
from pyspark.sql.window import *
from pyspark.sql.window import *
from delta.tables import *

current_dim_df.write.format('delta').mode('overwrite').saveAsTable('pyspark_cata_vijay.source.current_dim_df')

dlt_obj = DeltaTable.forName(spark, 'pyspark_cata_vijay.source.current_dim_df')


dlt_obj.alias('trg').merge(delta_df.alias('src'),'trg.customer_id = src.customer_id AND trg.is_current=True').whenMatchedUpdate(condition='src.name <> trg.name OR src.email<>trg.email OR src.city <> trg.city OR src.membership_tier <> trg.membership_tier',
set={'name': 'src.name', 'email': 'src.email', 'city': 'src.city', 'is_current': 'false', 'end_date': 'current_date()'}).whenNotMatchedInsertAll()


In [0]:
%sql
Select * from pyspark_cata_vijay.source.current_dim_df

## Scenario 10: Data Quality Monitoring & Quarantine

### Problem Statement:
Build a data quality framework that validates incoming order data and:

1. **Apply multiple validation rules:**
   * order_amount must be > 0 and < 100,000
   * order_date must not be in the future
   * customer_id must not be null
   * email must be valid format
   * quantity must be between 1 and 1000

2. **Classify records:**
   * PASS: All validations passed
   * QUARANTINE: 1-2 validation failures (can be reviewed)
   * REJECT: 3+ validation failures

3. **Output three datasets:**
   * Valid orders (PASS) → load into production table
   * Quarantine orders → send to data steward for review
   * Rejected orders → store in error table with failure reasons

4. **Generate quality metrics:**
   * Total records processed
   * Pass/Quarantine/Reject counts and percentages
   * Most common validation failures

### Sample Data:
Orders with various quality issues

In [0]:
from datetime import datetime, timedelta

# Generate sample orders with various quality issues
orders_data = [
    # Valid records
    ("ORD001", "CUST001", "alice@email.com", "2024-03-10", 150.50, 5, "Shipped"),
    ("ORD002", "CUST002", "bob@email.com", "2024-03-11", 2500.00, 10, "Delivered"),
    
    # Quarantine: 1 failure (invalid email)
    ("ORD003", "CUST003", "invalid-email", "2024-03-12", 500.00, 3, "Pending"),
    
    # Quarantine: 2 failures (negative amount + future date)
    ("ORD004", "CUST004", "david@email.com", "2025-01-01", -100.00, 2, "Pending"),
    
    # Reject: 3 failures (null customer_id + invalid email + zero amount)
    ("ORD005", None, "no-at-sign", "2024-03-13", 0.00, 5, "Pending"),
    
    # Quarantine: 1 failure (quantity too high)
    ("ORD006", "CUST006", "frank@email.com", "2024-03-13", 5000.00, 1500, "Pending"),
    
    # Valid record
    ("ORD007", "CUST007", "grace@email.com", "2024-03-14", 750.00, 8, "Shipped"),
    
    # Reject: 4 failures (null customer + invalid email + amount too high + quantity zero)
    ("ORD008", None, "bad.email", "2024-03-14", 150000.00, 0, "Pending"),
    
    # Quarantine: 2 failures (future date + invalid email format)
    ("ORD009", "CUST009", "user@", "2024-12-31", 300.00, 4, "Pending"),
    
    # Valid record
    ("ORD010", "CUST010", "henry@email.com", "2024-03-15", 1250.00, 15, "Processing"),
]

orders_df = spark.createDataFrame(
    orders_data,
    ["order_id", "customer_id", "email", "order_date", "order_amount", "quantity", "status"]
)

print("Raw Orders Data (with quality issues):")
display(orders_df)

print("\n📋 Validation Rules:")
print("  1. order_amount: 0 < amount < 100,000")
print("  2. order_date: not in the future")
print("  3. customer_id: not null")
print("  4. email: valid format (contains @ and domain)")
print("  5. quantity: 1 <= qty <= 1000")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
# orders_df.createOrReplaceTempView('orders_df')
orders_df = orders_df.withColumn('order_date',to_date(col('order_date')))
email_pattern = r'^[^@]+@[^@]+\.[^@]+$'

#Filtering out Valid Records
orders_df_valid = orders_df.filter((col('order_amount')>0) & (col('order_amount')<100000)).filter(col('order_date')<=current_date())\
    .filter(col('customer_id').isNotNull()).filter((col('quantity')>=1) & (col('quantity')<=1000))\
        .filter(col('email').rlike(email_pattern))

# Filtering Invalid Order Amount
in_valid_order_amount = orders_df.filter((col('order_amount')<=0) | (col('order_amount')>=100000)).withColumn('Reason',lit("Invalid Order Amount"))

# Filtering Invalid Order Date
in_valid_order_date = orders_df.filter(col('order_date')>current_date()).withColumn('Reason',lit("Invalid Order Date"))

# Filtering Invalid Customer ID
in_valid_customer_id = orders_df.filter(col('customer_id').isNull()).withColumn('Reason',lit("Invalid Customer ID"))

# Filtering Invalid Order quantity
in_valid_order_qty = orders_df.filter((col('quantity')<1) | (col('quantity')>1000)).withColumn('Reason',lit("Invalid Order Qty"))

#Filtering Invalid Emails
in_valid_emails = orders_df.filter(~col('email').rlike(email_pattern)).withColumn('Reason', lit("Invalid Email"))

#Combining all Invalid Records
df_invalid = in_valid_order_amount.union(in_valid_order_date).union(in_valid_customer_id).union(in_valid_order_qty).union(in_valid_emails)

#Orders which needs to be reviewed
df_quarantine = df_invalid.groupBy('order_id').agg(count(col('Reason')).alias('total_failures')).filter(col('total_failures')<=2)

#Rejected Orders 
df_rejected = df_invalid.groupBy('order_id').agg(count(col('Reason')).alias('total_failures')).filter(col('total_failures')>=3)

#Writing the Valid Records into Production table
orders_df_valid.write.format('delta').mode('overwrite').saveAsTable('sdp_catalog.target.orders_prod')

#Writing rejected records
df_invalid.write.format('delta').mode('overwrite').saveAsTable('sdp_catalog.target.orders_rejected_records')

## Scenario 11: Complex Window Functions - Running Totals & Rankings

### Problem Statement:
Analyze sales representative performance data and calculate:

1. **Running total** of sales amount for each rep (ordered by date)
2. **Moving average** of last 3 transactions for each rep
3. **Rank** reps within each region by total sales
4. **Dense rank** for reps by transaction count (handle ties properly)
5. **Lead/Lag**: Compare each transaction to the previous and next transaction amount for the same rep
6. **Percent of total**: Each rep's sales as % of their region's total
7. **First/Last value**: First and last transaction amount for each rep

### Business Goal:
Generate a comprehensive sales analytics report showing:
* Top performing reps per region
* Sales trends (are amounts increasing or decreasing?)
* Consistency metrics (moving averages)
* Regional contribution analysis

### Sample Data:
Sales transactions across different regions and reps

In [0]:
# Sales transactions by representatives across regions
sales_data = [
    # East Region
    ("REP001", "Alice", "East", "2024-03-01", 1000.0),
    ("REP001", "Alice", "East", "2024-03-05", 1500.0),
    ("REP001", "Alice", "East", "2024-03-10", 2000.0),
    ("REP001", "Alice", "East", "2024-03-15", 1200.0),
    
    ("REP002", "Bob", "East", "2024-03-02", 800.0),
    ("REP002", "Bob", "East", "2024-03-08", 900.0),
    ("REP002", "Bob", "East", "2024-03-12", 1100.0),
    
    # West Region
    ("REP003", "Carol", "West", "2024-03-03", 2500.0),
    ("REP003", "Carol", "West", "2024-03-07", 3000.0),
    ("REP003", "Carol", "West", "2024-03-11", 2800.0),
    ("REP003", "Carol", "West", "2024-03-14", 3200.0),
    
    ("REP004", "David", "West", "2024-03-04", 1500.0),
    ("REP004", "David", "West", "2024-03-09", 1600.0),
    
    # Central Region
    ("REP005", "Emma", "Central", "2024-03-01", 1800.0),
    ("REP005", "Emma", "Central", "2024-03-06", 2200.0),
    ("REP005", "Emma", "Central", "2024-03-12", 2000.0),
    
    ("REP006", "Frank", "Central", "2024-03-03", 1000.0),
    ("REP006", "Frank", "Central", "2024-03-10", 1100.0),
    ("REP006", "Frank", "Central", "2024-03-15", 950.0),
]

sales_reps_df = spark.createDataFrame(
    sales_data,
    ["rep_id", "rep_name", "region", "transaction_date", "sales_amount"]
)

print("Sales Representative Transactions:")
display(sales_reps_df.orderBy("region", "rep_id", "transaction_date"))

In [0]:
# Running total of sales amount for each rep (ordered by date)
# Moving average of last 3 transactions for each rep
# Rank reps within each region by total sales
# Dense rank for reps by transaction count (handle ties properly)
# Lead/Lag: Compare each transaction to the previous and next transaction amount for the same rep
# Percent of total: Each rep's sales as % of their region's total
# First/Last value: First and last transaction amount for each rep

from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

sales_reps_df_new = sales_reps_df.withColumn('cum_sales',sum('sales_amount').over(Window.partitionBy('rep_id').orderBy(col('transaction_date').asc())))\
    .withColumn('mov_avg',avg('sales_amount').over(Window.partitionBy('rep_id').orderBy(col('transaction_date').asc()).rowsBetween(-2,Window.currentRow))).withColumn('total_sales_rgn',sum('sales_amount').over(Window.partitionBy('region').orderBy(col('transaction_date').asc())))\
        .withColumn('rep_rank',row_number().over(Window.partitionBy('region').orderBy(col('total_sales_rgn').desc())))\
            .withColumn('tran_cnt',count('*').over(Window.partitionBy('rep_id')))\
                .withColumn('den_rank',dense_rank().over(Window.orderBy(col('tran_cnt').desc())))\
                    .withColumn('prev_tran_date',lag('transaction_date').over(Window.partitionBy('rep_id').orderBy(col('transaction_date').asc()))).withColumn('prev_sales_amt',lag('sales_amount').over(Window.partitionBy('rep_id').orderBy(col('transaction_date').asc())))
sales_reps_df_new.write.format('delta').mode('overwrite').saveAsTable('pyspark_cata_vijay.source.sales_reps_df_new')

In [0]:
%sql
Select * from pyspark_cata_vijay.source.sales_reps_df_new

In [0]:
%sql
With CTE1 as
(Select *,(sales_amount-prev_sales_amt) as comp,
row_number()over(partition by rep_id order by transaction_date asc) as rnk_first,
row_number()over(partition by rep_id order by transaction_date desc) as rnk_last
from pyspark_cata_vijay.source.sales_reps_df_new),
CTE2 as
(Select *,
case when rnk_first=1 then transaction_date end as first_transaction_date,
case when rnk_first=1 then sales_amount end as first_sales_amount,
case when rnk_last=1 then transaction_date end as last_transaction_date,
case when rnk_last=1 then sales_amount end as last_sales_amount
from CTE1),
CTE3 as
(Select *,sum(sales_amount)over(Partition by rep_id) as total_sales_per_rep,sum(sales_amount) over() as total_sales from CTE2),
CTE4 as
(Select *,round((total_sales_per_rep/total_sales)*100.0,2) as sales_percent from CTE3)
Select distinct rep_id,rep_name,region,cum_sales as running_total,mov_avg,rep_rank,den_rank,comp,
first_transaction_date,last_transaction_date,sales_percent,lead(sales_amount)over(Partition by rep_id order by transaction_date desc) as next_sales_amount from CTE4



## Scenario 12: Handling Skewed Data in Joins

### Problem Statement:
You have two datasets to join:
* **Orders**: Contains millions of orders (some customers have way more orders than others)
* **Customers**: Customer master data

The join is experiencing severe performance issues because:
* One customer ("CUST_VIP") has 60% of all orders (extreme skew)
* Standard join causes data skew – one executor processes most of the data

### Your Tasks:
1. **Identify the skew**: Show the distribution of orders per customer
2. **Implement salting technique**: 
   * Add a random salt to the skewed key
   * Replicate the dimension side
   * Join on salted keys
3. **Compare performance**: Measure execution time with and without salting
4. **Validate correctness**: Ensure both approaches produce the same results

### Salting Strategy:
* Identify customers with > 1000 orders as "skewed"
* Add salt suffix (0-9) to customer_id for skewed customers
* Replicate customer dimension records 10x (one for each salt value)
* Non-skewed customers: regular join

### Sample Data:
Small dataset representing the skew pattern

In [0]:
import random 
from pyspark.sql.functions import *
num_orders = random.randint(1, 5)

con = concat_ws(f"con,{num_orders}")

print(con)

In [0]:
import random 
from pyspark.sql.functions import col


# Simulate skewed orders (one customer has majority of orders)
skewed_orders = []

# VIP customer with massive number of orders (simulating 60% of data)
for i in range(1, 151):  # 150 orders from one customer
    skewed_orders.append(
        (f"ORD_{i}", "CUST_VIP", f"PROD{random.randint(1,10)}", 
         random.randint(1, 5), random.uniform(10, 500))
    )

print(skewed_orders)
# Regular customers with normal order counts
for cust_num in range(1, 21):  # 20 regular customers
    num_orders = random.randint(1, 5)  # Few orders each
    for ord in range(num_orders):
        skewed_orders.append(
            (f"ORD_{len(skewed_orders)+1:04d}", f"CUST_{cust_num:03d}", 
             f"PROD{random.randint(1,10):03d}", 
             random.randint(1, 3), random.uniform(10, 200))
        )

orders_skewed_df = spark.createDataFrame(
    skewed_orders,
    ["order_id", "customer_id", "product_id", "quantity", "amount"]
)

print(f"Total orders: {orders_skewed_df.count()}")
print("\nOrder distribution by customer:")
display(orders_skewed_df.groupBy("customer_id").count().orderBy(col("count").desc()))

In [0]:
# Customer dimension (much smaller table)
customers_data = [
    ("CUST_VIP", "VIP Corporation", "Platinum", "New York"),
]

# Add regular customers
for i in range(1, 21):
    tier = random.choice(["Gold", "Silver", "Bronze"])
    city = random.choice(["Chicago", "Boston", "Seattle", "Austin"])
    customers_data.append(
        (f"CUST_{i:03d}", f"Customer {i}", tier, city)
    )

customers_df = spark.createDataFrame(
    customers_data,
    ["customer_id", "customer_name", "tier", "city"]
)

print("Customer Master Data:")
display(customers_df)

print("\n⚠️ Challenge: CUST_VIP has 150 orders out of ~200 total (75% skew)")
print("This will cause one partition to process most data in a standard join!")